# Paper 3 — *Where Should LoRA Go? Component-Type Placement for Hybrid Language Models*

This notebook is a complete, checkpointed research pipeline for:

- **Model discovery and module-path verification** on **Qwen 3.5-0.8B** and **Falcon-H1-0.5B**
- **Selective LoRA placement** in hybrid architectures (6 conditions per model)
- **Training, evaluation, analysis, and publication outputs**
- **Checkpoint-based fault tolerance** — survives disconnects and resumes automatically

**All artifacts are written to** `ROOT_DIR` **(configurable in Section 0.2 below)**

> **Important:** Section 1 is intentionally designed to run **before any training**. Do **not** enable the full experiment grid until you have manually inspected the discovered module trees and verification outputs.


## Pre-implementation reasoning: reviewer-facing research questions, risks, and scope

This notebook bakes the following decisions into the design.

### 1) Additional research questions that make the paper stronger
**Core questions (must-answer for Findings):**
1. Does **attention-only** LoRA recover most of full-LoRA performance in hybrid models?
2. Does **linear/SSM-only** LoRA underperform attention-only LoRA, consistent with prior pure-SSM findings?
3. Is the answer **architecture-dependent**? Sequential hybridization (Qwen 3.5) may behave differently from parallel hybridization (Falcon-H1).
4. Is **MLP-only** a strong control, or does hybrid-specific routing matter more than generic FFN adaptation?
5. Which placement gives the best **accuracy / Colab compute unit** tradeoff?

**High-value extensions (implemented here as optional toggles):**
6. **Cross-task transfer:** train on one domain, evaluate all others. This directly tests whether softmax vs GDN/SSM adaptation transfers differently.
7. **Forgetting analysis:** measure performance drops on non-target benchmarks after fine-tuning.
8. **Value-only control:** optional condition aligned with the recent linear-attention theory that value-matrix-restricted updates preserve ICL better.
9. **Layer-selection diagnostic:** cheap **Layer Card proxy** that ranks layers/components by gradient sensitivity per parameter budget.
10. **Learning dynamics:** track LoRA norm growth by module during training.
11. **Rank interaction:** optional reduced ablation over `r ∈ {8, 16, 32}` on a small set of core conditions.
12. **Seed stability:** optional multi-seed reruns for the most important comparisons.

### 2) Cross-task transfer
Yes — this should be in the paper. The notebook makes it automatic:
- fine-tune on **general instruction**, **math**, or **code**
- evaluate on **MMLU, GSM8K, ARC-Challenge, HellaSwag, HumanEval**
- compute target-task gain **and** non-target transfer / forgetting

This is one of the clearest ways to show that different hybrid components specialize differently.

### 3) Layer Card diagnostic
Yes, but as a **proxy diagnostic**, not as the core claim. Re-implementing the full projected residual norm machinery exactly is expensive and not necessary for the first submission. This notebook includes a low-cost approximation:
- run a few calibration batches
- accumulate per-layer / per-component gradient sensitivity
- normalize by parameter count
- compare the ranking to the empirical winners

That gives a theory-facing analysis without blowing the Colab budget.

### 4) Learning dynamics
Yes. The notebook includes a callback that periodically records LoRA host-module norms. This supports statements such as:
- “even when all components are targetable, most adaptation mass accumulates in attention / MLP / GDN modules”
- “SSM-only adapters move, but not enough to match attention-only”

This also sets up Paper 4 nicely.

### 5) Rank × placement interaction
This is valuable, but should be **optional** for compute reasons.
- **Findings-ready:** keep rank fixed at `r=16`
- **Main-track upgrade:** reduced rank ablation on only the most informative conditions:
  - full baseline
  - attention-only / softmax-only
  - GDN-only / SSM-only

### 6) Forgetting analysis
Absolutely necessary. Reviewers will care whether the “best” placement is only best on the trained domain or also least destructive elsewhere. This notebook computes:
- `delta_vs_base` on every benchmark
- mean **non-target delta**
- a simple **forgetting score**

### 7) Methodological risks and mitigations
- **Base vs instruct confound:** use **base models** for the core study. Optionally add instruct sanity checks later.
- **PEFT targeting ambiguity:** never rely on short suffixes alone for hybrid placement. This notebook discovers exact module paths and uses **exact dotted module names** as LoRA targets.
- **Falcon-H1 LoRA constraints:** hard-exclude forbidden modules and never merge Falcon adapters back into the base model.
- **Hyperparameter sensitivity:** keep the main grid fixed; use optional small ablations if time remains.
- **Benchmark contamination:** acknowledge it. The notebook uses consistent held-out evaluation subsets and fixed few-shot pools, but contamination cannot be fully ruled out for standard public benchmarks.

### 8) Findings vs main-track
**Minimum credible Findings paper:**
- Qwen 3.5-0.8B Base + Falcon-H1-0.5B Base
- all 6 placement conditions for each
- 3 fine-tuning domains
- full benchmark suite
- trainable parameter counts, time, CU, efficiency, forgetting
- discovery + verification section showing exact module targeting

**What likely upgrades toward main track:**
- optional Falcon-H1-1.5B scale point
- multi-seed core comparisons
- rank ablation
- Layer Card proxy + dynamics analysis
- optional value-only theoretical control
- possibly one more hybrid family if compute permits

### Notebook default policy
This notebook defaults to a **Findings-first** configuration:
- all core conditions
- one seed
- quick/fail-safe preset
- optional diagnostics present but disabled by default


In [ ]:
# SECTION 0.1 — Install dependencies
# What this cell does:
# - installs the latest Transformers/PEFT/TRL from source
# - installs datasets/evaluate/bitsandbytes and notebook utilities
#
# Expected output:
# - pip install logs
# - a final confirmation line

import os
import sys
import subprocess
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"

def pip_install(packages):
    cmd = [sys.executable, "-m", "pip", "install", "-qU", *packages]
    print("Installing:", " ".join(packages))
    subprocess.check_call(cmd)

packages = [
    "git+https://github.com/huggingface/transformers.git@main",
    "git+https://github.com/huggingface/peft.git@main",
    "git+https://github.com/huggingface/trl.git@main",
    "datasets",
    "evaluate",
    "accelerate",
    "bitsandbytes",
    "scikit-learn",
    "pandas",
    "matplotlib",
    "tabulate",
    "sentencepiece",
    "protobuf",
    "safetensors",
    "jsonlines",
    "packaging",
]
pip_install(packages)

print("✅ Dependency installation complete.")


In [ ]:
# SECTION 0.2 — Storage setup and project directory tree
# What this cell does:
# - sets the project root directory (configurable)
# - creates all required output folders
#
# Expected output:
# - project root path and folder listing
#
# ─── CONFIGURE THIS ──────────────────────────────────────────────────────────
# Set ROOT_DIR to wherever you want outputs saved.
# Examples:
#   - Google Colab + Drive:  Path("/content/drive/MyDrive/lora-placement")
#   - RunPod / cloud:       Path("/workspace/lora-placement")
#   - Local machine:        Path.home() / "lora-placement"
# ────────────────────────────────────────────────────────────────────────────

import os
from pathlib import Path

ROOT_DIR = Path.home() / "lora-placement"   # ← CHANGE THIS TO YOUR PREFERRED PATH

# Optional: uncomment to mount Google Drive on Colab
# from google.colab import drive
# drive.mount("/content/drive", force_remount=False)
# ROOT_DIR = Path("/content/drive/MyDrive/lora-placement")

CHECKPOINTS_DIR = ROOT_DIR / "checkpoints"
RESULTS_DIR = ROOT_DIR / "results"
FIGURES_DIR = ROOT_DIR / "figures"
TABLES_DIR = ROOT_DIR / "tables"
LOGS_DIR = ROOT_DIR / "logs"
MODELS_DIR = ROOT_DIR / "models"
DATA_CACHE_DIR = ROOT_DIR / "data_cache"
MODULE_TREES_DIR = ROOT_DIR / "module_trees"
ANALYSIS_CACHE_DIR = ROOT_DIR / "analysis_cache"
EVAL_DETAILS_DIR = RESULTS_DIR / "eval_details"
DISCOVERY_DIR = RESULTS_DIR / "discovery"
SUMMARY_DIR = RESULTS_DIR / "summary"

for p in [ROOT_DIR, CHECKPOINTS_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR,
          LOGS_DIR, MODELS_DIR, DATA_CACHE_DIR, MODULE_TREES_DIR,
          ANALYSIS_CACHE_DIR, EVAL_DETAILS_DIR, DISCOVERY_DIR, SUMMARY_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f"📁 Project root: {ROOT_DIR}")
for p in [CHECKPOINTS_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR, LOGS_DIR, MODELS_DIR]:
    print(f"   - {p.relative_to(ROOT_DIR)}")


In [ ]:
# SECTION 0.3 — Imports, global configuration, reproducibility, and environment snapshot
# What this cell does:
# - imports all runtime dependencies
# - defines experiment presets
# - configures precision and reproducibility
# - records a full environment snapshot to Drive
#
# Expected output:
# - GPU / package version summary
# - chosen preset and precision mode

import os
import re
import gc
import io
import sys
import json
import time
import math
import copy
import pickle
import random
import shutil
import inspect
import textwrap
import traceback
import importlib
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Iterable
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn

import datasets
from datasets import load_dataset, load_from_disk, Dataset, DatasetDict

import evaluate

import transformers
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainerCallback,
    set_seed as hf_set_seed,
)
from transformers.trainer_utils import get_last_checkpoint

import peft
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

import trl
from trl import SFTTrainer, SFTConfig

SEED = 3407

def seed_everything(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    hf_set_seed(seed)

seed_everything(SEED)

# GPU cost rates (approximate cost/h)

PRESETS = {
    "quick_findings": {
        "models": ["qwen3_5_0_8b_base", "falcon_h1_0_5b_base"],
        "datasets": ["ultrachat", "gsm8k_train", "codealpaca"],
        "max_train_samples_by_dataset": {
            "ultrachat": 2000,
            "gsm8k_train": 2000,
            "codealpaca": 2000,
        },
        "max_eval_samples_by_benchmark": {
            "mmlu": 256,
            "gsm8k": 256,
            "arc_challenge": 256,
            "hellaswag": 256,
            "humaneval": 64,
        },
    },
    "full_findings": {
        "models": ["qwen3_5_0_8b_base", "falcon_h1_0_5b_base"],
        "datasets": ["ultrachat", "gsm8k_train", "codealpaca"],
        "max_train_samples_by_dataset": {
            "ultrachat": 5000,
            "gsm8k_train": 7500,
            "codealpaca": 10000,
        },
        "max_eval_samples_by_benchmark": {
            "mmlu": 1024,
            "gsm8k": 1024,
            "arc_challenge": 1024,
            "hellaswag": 1024,
            "humaneval": 164,
        },
    },
    "q2_submission": {
        "models": ["qwen3_5_0_8b_base", "falcon_h1_0_5b_base"],
        "datasets": ["ultrachat", "gsm8k_train", "codealpaca"],
        "max_train_samples_by_dataset": {
            "ultrachat": 2000,
            "gsm8k_train": 2000,
            "codealpaca": 2000,
        },
        "max_eval_samples_by_benchmark": {
            "mmlu": 512,
            "gsm8k": 512,
            "arc_challenge": 512,
            "hellaswag": 512,
            "humaneval": 164,
        },
    },
    "main_track": {
        "models": ["qwen3_5_0_8b_base", "falcon_h1_0_5b_base", "falcon_h1_1_5b_base"],
        "datasets": ["ultrachat", "gsm8k_train", "codealpaca"],
        "max_train_samples_by_dataset": {
            "ultrachat": 8000,
            "gsm8k_train": 7500,
            "codealpaca": 10000,
        },
        "max_eval_samples_by_benchmark": {
            "mmlu": 2048,
            "gsm8k": 1319,
            "arc_challenge": 1172,
            "hellaswag": 2048,
            "humaneval": 164,
        },
    },
}

EXPERIMENT_PRESET = "q2_submission"  # ← Q2 paper: 2K training, 512 eval
CFG = PRESETS[EXPERIMENT_PRESET]

MODELS_TO_RUN = CFG["models"]
DATASETS_TO_USE = CFG["datasets"]
MAX_TRAIN_SAMPLES_BY_DATASET = CFG["max_train_samples_by_dataset"]
MAX_EVAL_SAMPLES_BY_BENCHMARK = CFG["max_eval_samples_by_benchmark"]

# Core training hyperparameters
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
PER_DEVICE_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
EFFECTIVE_BATCH_SIZE = PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
WEIGHT_DECAY = 0.01
MAX_SEQ_LENGTH = 1024
SAVE_STEPS = 200
LOGGING_STEPS = 10
EVAL_EVERY_N_STEPS = 200
LORA_DYNAMICS_EVERY_N_STEPS = 50

# Manual eval defaults
FEWSHOT_COUNTS = {
    "mmlu": 5,
    "gsm8k": 8,
    "arc_challenge": 25,
    "hellaswag": 10,
}
MAX_EVAL_PROMPT_TOKENS = 896
GSM8K_MAX_NEW_TOKENS = 256
HUMANEVAL_MAX_NEW_TOKENS = 256

# Runtime policy
AUTO_RUN_EXPERIMENT_GRID = False
RUN_BASELINE_EVALS_FIRST = True
RUN_INLINE_EVAL_AFTER_TRAIN = True
RUN_HUMANEVAL = True
RUN_HUMANEVAL_ONLY_IF_CODE_TRAIN = True
EVAL_DURING_TRAINING = True

# Optional extensions
ENABLE_VALUE_ONLY_CONDITION = False
ENABLE_LAYERCARD_PROXY = True
ENABLE_LORA_DYNAMICS_TRACKING = True
ENABLE_RANK_ABLATION = False
ENABLE_MULTI_SEED_CORE = False
ENABLE_INSTRUCT_SANITY = False
ENABLE_TARGET_PARAMETERS = False
RANK_ABLATION_RANKS = [8, 16, 32]
CORE_SEEDS = [SEED]
OPTIONAL_MULTI_SEEDS = [3407, 1337, 2026]

def get_device_name() -> str:
    if not torch.cuda.is_available():
        return "CPU"
    return torch.cuda.get_device_name(0)

def infer_colab_cu_rate(gpu_name: str) -> Optional[float]:
    # CU rate auto-detection removed for portability
    # Set ESTIMATED_CU_RATE manually in config above if needed
    return None

GPU_NAME = get_device_name()
IS_BF16_SUPPORTED = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
USE_BF16 = IS_BF16_SUPPORTED
USE_FP16 = bool(torch.cuda.is_available() and not USE_BF16)
DEFAULT_TORCH_DTYPE = torch.bfloat16 if USE_BF16 else (torch.float16 if USE_FP16 else torch.float32)
ESTIMATED_CU_RATE = infer_colab_cu_rate(GPU_NAME)

torch.backends.cuda.matmul.allow_tf32 = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

MODEL_REGISTRY = {
    "qwen3_5_0_8b_base": {
        "hf_id": "Qwen/Qwen3.5-0.8B-Base",
        "family": "qwen3_5",
        "variant": "base",
        "sequential_or_parallel": "sequential",
        "expected_num_layers": 24,
        "notes": "Hybrid 3:1 Gated DeltaNet / full-attention layout.",
    },
    "falcon_h1_0_5b_base": {
        "hf_id": "tiiuae/Falcon-H1-0.5B-Base",
        "family": "falcon_h1",
        "variant": "base",
        "sequential_or_parallel": "parallel",
        "expected_num_layers": 36,
        "notes": "Parallel hybrid attention + Mamba-2 SSM heads.",
    },
    "falcon_h1_1_5b_base": {
        "hf_id": "tiiuae/Falcon-H1-1.5B-Base",
        "family": "falcon_h1",
        "variant": "base",
        "sequential_or_parallel": "parallel",
        "expected_num_layers": 24,
        "notes": "Optional scale point.",
    },
    "qwen3_5_0_8b_instruct": {
        "hf_id": "Qwen/Qwen3.5-0.8B",
        "family": "qwen3_5",
        "variant": "instruct",
        "sequential_or_parallel": "sequential",
        "expected_num_layers": 24,
        "notes": "Optional instruct sanity check.",
    },
    "falcon_h1_0_5b_instruct": {
        "hf_id": "tiiuae/Falcon-H1-0.5B-Instruct",
        "family": "falcon_h1",
        "variant": "instruct",
        "sequential_or_parallel": "parallel",
        "expected_num_layers": 36,
        "notes": "Optional instruct sanity check.",
    },
}

def json_default(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if isinstance(obj, set):
        return sorted(obj)
    return str(obj)

def sanitize_key(s: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_.-]+", "_", s)

class CheckpointManager:
    def __init__(self, root: Path):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)

    def _path(self, key: str, ext: str = "pkl") -> Path:
        return self.root / f"{sanitize_key(key)}.{ext}"

    def save(self, key: str, obj: Any) -> str:
        path = self._path(key, "pkl")
        tmp = path.with_suffix(path.suffix + ".tmp")
        with open(tmp, "wb") as f:
            pickle.dump(obj, f)
        os.replace(tmp, path)
        return str(path)

    def load(self, key: str) -> Any:
        path = self._path(key, "pkl")
        if not path.exists():
            return None
        with open(path, "rb") as f:
            return pickle.load(f)

    def save_json(self, key: str, obj: Any) -> str:
        path = self._path(key, "json")
        tmp = path.with_suffix(path.suffix + ".tmp")
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(obj, f, indent=2, ensure_ascii=False, default=json_default)
        os.replace(tmp, path)
        return str(path)

    def load_json(self, key: str) -> Any:
        path = self._path(key, "json")
        if not path.exists():
            return None
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)

    def list_paths(self, suffix: str = ".pkl") -> List[Path]:
        return sorted(self.root.glob(f"*{suffix}"))

    def exists(self, key: str, ext: str = "pkl") -> bool:
        return self._path(key, ext).exists()

    @staticmethod
    def append_jsonl(path: Path, row: Dict[str, Any]):
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "a", encoding="utf-8") as f:
            f.write(json.dumps(row, ensure_ascii=False, default=json_default) + "\n")

ckpt = CheckpointManager(CHECKPOINTS_DIR)

def save_text(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
    os.replace(tmp, path)

def maybe_load_pickle(path: Path):
    if not path.exists():
        return None
    with open(path, "rb") as f:
        return pickle.load(f)

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

env_snapshot = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "python": sys.version,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "trl": trl.__version__,
    "datasets": datasets.__version__,
    "gpu_name": GPU_NAME,
    "cuda_available": torch.cuda.is_available(),
    "bf16_supported": IS_BF16_SUPPORTED,
    "use_bf16": USE_BF16,
    "use_fp16": USE_FP16,
    "default_torch_dtype": str(DEFAULT_TORCH_DTYPE),
    "estimated_cu_rate": ESTIMATED_CU_RATE,
    "experiment_preset": EXPERIMENT_PRESET,
    "models_to_run": MODELS_TO_RUN,
    "datasets_to_use": DATASETS_TO_USE,
}

save_text(LOGS_DIR / "environment_snapshot.json", json.dumps(env_snapshot, indent=2, default=json_default))

print("Environment snapshot:")
print(json.dumps(env_snapshot, indent=2, default=json_default))


# Section 1 — Model loading, module discovery, condition construction, and verification

This is the most important part of the notebook.

**Why it exists:** PEFT target matching is global. In hybrid models, that is dangerous. We must:
1. load the actual model,
2. discover the true module tree,
3. map modules to component types,
4. build condition-specific exact module target lists,
5. verify that LoRA is attached only where intended.

This section writes:
- full module trees
- per-layer child summaries
- condition-to-target-module manifests
- verification reports
- trainable-parameter tables and figures


In [ ]:
# SECTION 1.1 — Model/tokenizer loading helpers and structural discovery utilities
# What this cell does:
# - defines robust loaders for Qwen 3.5 and Falcon-H1
# - discovers the decoder-layer container dynamically
# - builds a richly annotated module manifest
#
# Expected output:
# - function definitions only

def resolve_text_config(config):
    return getattr(config, "text_config", config)

def load_tokenizer(model_key: str):
    model_id = MODEL_REGISTRY[model_key]["hf_id"]
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token
    tokenizer.padding_side = "right"
    return tokenizer

def load_model(model_key: str, training: bool = False):
    model_id = MODEL_REGISTRY[model_key]["hf_id"]
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        trust_remote_code=True,
        torch_dtype=DEFAULT_TORCH_DTYPE,
        low_cpu_mem_usage=True,
    )
    if training:
        model.config.use_cache = False
        try:
            model.gradient_checkpointing_enable()
        except Exception:
            pass
        if hasattr(model, "enable_input_require_grads"):
            try:
                model.enable_input_require_grads()
            except Exception:
                pass
    else:
        model.eval()
    return model

def load_model_and_tokenizer(model_key: str, training: bool = False):
    tokenizer = load_tokenizer(model_key)
    model = load_model(model_key, training=training)
    if getattr(model.config, "pad_token_id", None) is None and tokenizer.pad_token_id is not None:
        model.config.pad_token_id = tokenizer.pad_token_id
    return model, tokenizer

def get_layer_types_from_config(config) -> List[str]:
    cfg = resolve_text_config(config)
    for attr in ["layer_types", "layers_block_type", "block_types"]:
        if hasattr(cfg, attr):
            value = getattr(cfg, attr)
            if value is None:
                continue
            try:
                return list(value)
            except Exception:
                pass
    return []

def find_decoder_layers_container(model) -> Tuple[str, nn.ModuleList]:
    cfg = resolve_text_config(model.config)
    expected = getattr(cfg, "num_hidden_layers", None)
    candidates = []
    for name, module in model.named_modules():
        if isinstance(module, nn.ModuleList):
            length = len(module)
            score = 0
            if expected is not None and length == expected:
                score += 100
            if name.endswith("layers"):
                score += 20
            if ".layers" in name:
                score += 10
            lname = name.lower()
            if any(bad in lname for bad in ["vision", "visual", "audio", "encoder", "decoder.layers.layers"]):
                score -= 50
            if score > 0:
                candidates.append((score, name, module))
    if not candidates:
        raise ValueError("Could not find decoder layer ModuleList dynamically.")
    candidates.sort(reverse=True, key=lambda x: x[0])
    _, best_name, best_module = candidates[0]
    return best_name, best_module

def count_direct_params(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters(recurse=False))

def direct_param_shapes(module: nn.Module) -> Dict[str, List[int]]:
    out = {}
    for name, p in module.named_parameters(recurse=False):
        out[name] = list(p.shape)
    return out

def module_is_linear_like(module: nn.Module) -> bool:
    if isinstance(module, nn.Linear):
        return True
    cls = module.__class__.__name__.lower()
    if "linear8bitlt" in cls or "linear4bit" in cls:
        return True
    if "linear" in cls and hasattr(module, "weight"):
        weight = getattr(module, "weight")
        if hasattr(weight, "ndim") and int(weight.ndim) == 2:
            return True
    return False

def module_is_conv1d_like(module: nn.Module) -> bool:
    if isinstance(module, nn.Conv1d):
        return True
    cls = module.__class__.__name__.lower()
    return "conv1d" in cls or "causalconv1d" in cls

_LAYER_RE = re.compile(r"\.layers\.(\d+)\.")
_LAYER_RE_END = re.compile(r"\.layers\.(\d+)$")

def extract_layer_idx_from_name(name: str) -> Optional[int]:
    m = _LAYER_RE.search(name)
    if m:
        return int(m.group(1))
    m = _LAYER_RE_END.search(name)
    if m:
        return int(m.group(1))
    return None

def infer_top_component_qwen(child_name: str, child: nn.Module, layer_type: str) -> Optional[str]:
    lname = child_name.lower()
    cls = child.__class__.__name__.lower()
    if any(tok in lname for tok in ["mlp", "ffn", "feed_forward"]) or "mlp" in cls:
        return "mlp"
    if any(tok in lname for tok in ["attn", "attention"]) or any(tok in cls for tok in ["attn", "attention", "delta", "gated"]):
        if "full" in layer_type or "attention" in layer_type:
            return "softmax_attn"
        if "linear" in layer_type or "delta" in cls or "gated" in cls:
            return "gdn"
    if child_name == "self_attn":
        if "full" in layer_type or "attention" in layer_type:
            return "softmax_attn"
        if "linear" in layer_type:
            return "gdn"
    return None

def infer_top_component_falcon(child_name: str, child: nn.Module) -> Optional[str]:
    lname = child_name.lower()
    cls = child.__class__.__name__.lower()
    if any(tok in lname for tok in ["mlp", "ffn", "feed_forward"]) or any(tok in cls for tok in ["mlp", "ffn", "feedforward"]):
        return "mlp"
    if any(tok in lname for tok in ["attn", "attention"]) or any(tok in cls for tok in ["attn", "attention"]):
        return "attention"
    if any(tok in lname for tok in ["mamba", "ssm"]) or any(tok in cls for tok in ["mamba", "ssm"]):
        return "ssm"
    return None

def generic_component_fallback(model_family: str, full_name: str, short_name: str, class_name: str, layer_type: Optional[str]) -> str:
    fname = full_name.lower()
    sname = short_name.lower()
    cname = class_name.lower()

    if any(tok in fname for tok in ["embed_tokens", "wte", "word_embeddings", "tok_embeddings"]):
        return "embedding"
    if "lm_head" in fname:
        return "lm_head"
    if "norm" in fname or "layernorm" in cname or "rmsnorm" in cname:
        return "norm"

    if model_family == "qwen3_5":
        if ".mlp." in fname:
            return "mlp"
        if "linear" in (layer_type or ""):
            if ".self_attn." in fname or ".linear_attn." in fname:
                return "gdn"
        if "full" in (layer_type or "") or "attention" in (layer_type or ""):
            if ".self_attn." in fname or ".attention." in fname:
                return "softmax_attn"

    if model_family == "falcon_h1":
        if any(tok in fname for tok in [".mlp.", ".ffn.", ".feed_forward."]):
            return "mlp"
        if any(tok in sname for tok in ["q_proj", "k_proj", "v_proj", "o_proj"]) or "attention" in fname or "attn" in fname:
            return "attention"
        if any(tok in sname for tok in ["in_proj", "conv1d", "out_proj", "x_proj", "dt_proj"]) or "mamba" in fname or "ssm" in fname:
            return "ssm"

    return "other"

def discover_model_structure(model_key: str, force: bool = False) -> Dict[str, Any]:
    cache_key = f"discovery__{model_key}"
    if not force:
        cached = ckpt.load(cache_key)
        if cached is not None:
            print(f"✅ Loaded cached discovery for {model_key}")
            return cached

    model, tokenizer = load_model_and_tokenizer(model_key, training=False)
    family = MODEL_REGISTRY[model_key]["family"]
    layer_types = get_layer_types_from_config(model.config)
    layer_container_name, layers = find_decoder_layers_container(model)

    full_tree_lines = []
    for full_name, module in model.named_modules():
        dcount = count_direct_params(module)
        dshapes = direct_param_shapes(module)
        full_tree_lines.append(
            f"{full_name or '<root>'} :: {module.__class__.__name__} :: direct_params={dcount} :: shapes={dshapes}"
        )
    save_text(MODULE_TREES_DIR / f"{model_key}__named_modules.txt", "\n".join(full_tree_lines))

    layer_child_lines = []
    layer_prefix_component = {}

    for idx, layer in enumerate(layers):
        layer_type = layer_types[idx] if idx < len(layer_types) else "unknown"
        layer_child_lines.append(f"Layer {idx} [{layer_type}] :: {layer.__class__.__name__}")
        for child_name, child in layer.named_children():
            child_params = sum(p.numel() for p in child.parameters())
            layer_child_lines.append(f"  {child_name} :: {child.__class__.__name__} :: params={child_params}")
            if family == "qwen3_5":
                comp = infer_top_component_qwen(child_name, child, str(layer_type))
            elif family == "falcon_h1":
                comp = infer_top_component_falcon(child_name, child)
            else:
                comp = None
            if comp is not None:
                layer_prefix_component[f"{layer_container_name}.{idx}.{child_name}"] = comp

            for subname, subchild in child.named_children():
                sub_params = sum(p.numel() for p in subchild.parameters())
                desc = f"    {child_name}.{subname} :: {subchild.__class__.__name__} :: params={sub_params}"
                if module_is_linear_like(subchild) and hasattr(subchild, "in_features") and hasattr(subchild, "out_features"):
                    desc += f" :: Linear({subchild.in_features}, {subchild.out_features})"
                elif module_is_conv1d_like(subchild):
                    desc += " :: Conv1d [EXCLUDE FOR FALCON-H1]"
                layer_child_lines.append(desc)

    save_text(MODULE_TREES_DIR / f"{model_key}__layer_children.txt", "\n".join(layer_child_lines))
    sorted_prefixes = sorted(layer_prefix_component.items(), key=lambda x: len(x[0]), reverse=True)

    records = []
    for full_name, module in model.named_modules():
        short_name = full_name.split(".")[-1] if full_name else "<root>"
        class_name = module.__class__.__name__
        layer_idx = extract_layer_idx_from_name(full_name)
        layer_type = layer_types[layer_idx] if (layer_idx is not None and layer_idx < len(layer_types)) else None

        component = None
        for prefix, comp in sorted_prefixes:
            if full_name == prefix or full_name.startswith(prefix + "."):
                component = comp
                break
        if component is None:
            component = generic_component_fallback(family, full_name, short_name, class_name, layer_type)

        exclude_reason = None
        lora_eligible = module_is_linear_like(module)

        if family == "falcon_h1":
            if short_name == "conv1d" or module_is_conv1d_like(module):
                lora_eligible = False
                exclude_reason = "official_falcon_h1_exclude_conv1d"
            if short_name == "out_proj":
                lora_eligible = False
                exclude_reason = "official_falcon_h1_exclude_out_proj"

        target_parameter_candidates = []
        if ENABLE_TARGET_PARAMETERS and not lora_eligible:
            for pname, p in module.named_parameters(recurse=False):
                if p.ndim >= 2:
                    target_parameter_candidates.append(f"{full_name}.{pname}" if full_name else pname)

        records.append({
            "full_name": full_name,
            "short_name": short_name,
            "class_name": class_name,
            "layer_idx": layer_idx,
            "layer_type": layer_type,
            "component": component,
            "direct_param_count": int(count_direct_params(module)),
            "direct_param_shapes": direct_param_shapes(module),
            "lora_eligible": bool(lora_eligible),
            "exclude_reason": exclude_reason,
            "target_parameter_candidates": target_parameter_candidates,
        })

    manifest = {
        "model_key": model_key,
        "hf_id": MODEL_REGISTRY[model_key]["hf_id"],
        "family": family,
        "variant": MODEL_REGISTRY[model_key]["variant"],
        "layer_container_name": layer_container_name,
        "num_layers": len(layers),
        "layer_types": layer_types,
        "records": records,
        "layer_prefix_component": layer_prefix_component,
        "tokenizer_class": tokenizer.__class__.__name__,
        "model_class": model.__class__.__name__,
    }

    save_text(DISCOVERY_DIR / f"{model_key}__manifest.json", json.dumps(manifest, indent=2, default=json_default))
    ckpt.save(cache_key, manifest)

    clear_memory()
    del model, tokenizer
    clear_memory()

    print(f"✅ Discovery completed for {model_key}")
    print("Layer container:", layer_container_name)
    print("Num layers:", len(layers))
    print("Unique layer types:", sorted(set(layer_types)) if layer_types else "[none discovered]")
    return manifest


In [ ]:
# SECTION 1.2 — Condition construction from the discovered manifest
# What this cell does:
# - converts the manifest into exact target-module lists for each placement condition
# - uses full dotted module names (not short suffixes) for precise targeting
# - supports optional value-only and experimental target-parameter conditions
#
# Expected output:
# - function definitions only

ATTN_QKV_TOKENS = ["q_proj", "k_proj", "v_proj", "qkv", "wqkv", "query", "key", "value", "in_proj"]
ATTN_OUTPUT_TOKENS = ["o_proj", "c_proj", "proj_out", "output_proj"]
MLP_TOKENS = ["gate_proj", "up_proj", "down_proj", "fc1", "fc2", "wi", "wo", "w1", "w2", "w3", "c_fc", "c_proj"]
FORBIDDEN_FALCON_SHORT_NAMES = {"conv1d", "out_proj"}

def _contains_any(name: str, tokens: List[str]) -> bool:
    lname = name.lower()
    return any(tok in lname for tok in tokens)

def select_component_modules(manifest: Dict[str, Any], component: str) -> List[Dict[str, Any]]:
    out = []
    for r in manifest["records"]:
        if r["component"] != component:
            continue
        if not r["lora_eligible"]:
            continue
        out.append(r)
    return out

def select_qwen_softmax_modules(manifest: Dict[str, Any]) -> List[str]:
    records = select_component_modules(manifest, "softmax_attn")
    selected = [r["full_name"] for r in records if _contains_any(r["short_name"], ATTN_QKV_TOKENS + ATTN_OUTPUT_TOKENS)]
    if not selected:
        selected = [r["full_name"] for r in records]
    return sorted(set(selected))

def select_qwen_gdn_modules(manifest: Dict[str, Any]) -> List[str]:
    records = select_component_modules(manifest, "gdn")
    return sorted(set(r["full_name"] for r in records))

def select_mlp_modules(manifest: Dict[str, Any]) -> List[str]:
    records = select_component_modules(manifest, "mlp")
    selected = [r["full_name"] for r in records if _contains_any(r["short_name"], MLP_TOKENS)]
    if not selected:
        selected = [r["full_name"] for r in records]
    return sorted(set(selected))

def select_falcon_attention_modules(manifest: Dict[str, Any], include_output: bool = False) -> List[str]:
    records = select_component_modules(manifest, "attention")
    selected = []
    for r in records:
        short_name = r["short_name"]
        if _contains_any(short_name, ATTN_QKV_TOKENS):
            selected.append(r["full_name"])
        elif include_output and _contains_any(short_name, ATTN_OUTPUT_TOKENS):
            selected.append(r["full_name"])
    if not selected:
        selected = [r["full_name"] for r in records if r["short_name"] not in FORBIDDEN_FALCON_SHORT_NAMES]
    return sorted(set(selected))

def select_falcon_ssm_modules(manifest: Dict[str, Any]) -> List[str]:
    records = select_component_modules(manifest, "ssm")
    selected = []
    for r in records:
        if r["short_name"] in FORBIDDEN_FALCON_SHORT_NAMES:
            continue
        selected.append(r["full_name"])
    return sorted(set(selected))

def build_condition_specs(manifest: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    model_key = manifest["model_key"]
    family = manifest["family"]
    specs = {}

    if family == "qwen3_5":
        softmax = select_qwen_softmax_modules(manifest)
        gdn = select_qwen_gdn_modules(manifest)
        mlp = select_mlp_modules(manifest)

        specs["all_layers"] = {
            "target_modules": sorted(set(softmax + gdn + mlp)),
            "target_parameters": [],
            "notes": "Full LoRA baseline across softmax attention, GDN, and MLP modules.",
        }
        specs["softmax_only"] = {
            "target_modules": softmax,
            "target_parameters": [],
            "notes": "Softmax-attention projections only in full-attention layers.",
        }
        specs["gdn_only"] = {
            "target_modules": gdn,
            "target_parameters": [],
            "notes": "GDN / linear-attention component only.",
        }
        specs["mlp_only"] = {
            "target_modules": mlp,
            "target_parameters": [],
            "notes": "MLP-only control.",
        }
        specs["softmax_plus_mlp"] = {
            "target_modules": sorted(set(softmax + mlp)),
            "target_parameters": [],
            "notes": "Practical config: attention + MLP, skip GDN.",
        }
        specs["gdn_plus_mlp"] = {
            "target_modules": sorted(set(gdn + mlp)),
            "target_parameters": [],
            "notes": "Complementary config: GDN + MLP, skip softmax attention.",
        }

        if ENABLE_VALUE_ONLY_CONDITION:
            value_only = [m for m in softmax if "v_proj" in m.lower() or "value" in m.lower()]
            if value_only:
                specs["value_only"] = {
                    "target_modules": sorted(set(value_only)),
                    "target_parameters": [],
                    "notes": "Optional theory-aligned value-only condition.",
                }

    elif family == "falcon_h1":
        attention_qkv = select_falcon_attention_modules(manifest, include_output=False)
        attention_with_output = select_falcon_attention_modules(manifest, include_output=True)
        ssm = select_falcon_ssm_modules(manifest)
        mlp = select_mlp_modules(manifest)

        all_eligible = []
        for r in manifest["records"]:
            if not r["lora_eligible"]:
                continue
            if r["short_name"] in FORBIDDEN_FALCON_SHORT_NAMES:
                continue
            all_eligible.append(r["full_name"])

        specs["all_eligible"] = {
            "target_modules": sorted(set(all_eligible)),
            "target_parameters": [],
            "notes": "Full LoRA baseline respecting Falcon-H1 exclusions.",
        }
        specs["attention_only"] = {
            "target_modules": attention_qkv,
            "target_parameters": [],
            "notes": "Attention Q/K/V branch only.",
        }
        specs["ssm_only"] = {
            "target_modules": ssm,
            "target_parameters": [],
            "notes": "SSM-branch eligible modules only; excludes conv1d and out_proj.",
        }
        specs["mlp_only"] = {
            "target_modules": mlp,
            "target_parameters": [],
            "notes": "MLP-only control.",
        }
        specs["attention_plus_mlp"] = {
            "target_modules": sorted(set(attention_qkv + mlp)),
            "target_parameters": [],
            "notes": "Attention + MLP, skip SSM.",
        }
        specs["ssm_plus_mlp"] = {
            "target_modules": sorted(set(ssm + mlp)),
            "target_parameters": [],
            "notes": "SSM + MLP, skip attention.",
        }

        if ENABLE_VALUE_ONLY_CONDITION:
            value_only = [m for m in attention_with_output if "v_proj" in m.lower() or "value" in m.lower()]
            if value_only:
                specs["value_only"] = {
                    "target_modules": sorted(set(value_only)),
                    "target_parameters": [],
                    "notes": "Optional theory-aligned value-only attention condition.",
                }

    else:
        raise ValueError(f"Unsupported family: {family}")

    if ENABLE_TARGET_PARAMETERS:
        raw_param_candidates = []
        for r in manifest["records"]:
            raw_param_candidates.extend(r.get("target_parameter_candidates", []))
        raw_param_candidates = sorted(set(raw_param_candidates))
        if raw_param_candidates:
            specs["all_target_parameters_experimental"] = {
                "target_modules": [],
                "target_parameters": raw_param_candidates,
                "notes": "Experimental: apply LoRA via target_parameters where supported.",
            }

    for condition_name, spec in specs.items():
        spec["target_modules"] = sorted(set(spec.get("target_modules", [])))
        spec["target_parameters"] = sorted(set(spec.get("target_parameters", [])))
        if not spec["target_modules"] and not spec["target_parameters"]:
            raise ValueError(
                f"Condition {condition_name} for {model_key} ended up empty. "
                f"Inspect the discovery manifest and update the selection heuristics."
            )

    save_text(
        DISCOVERY_DIR / f"{model_key}__condition_specs.json",
        json.dumps(specs, indent=2, default=json_default),
    )
    return specs

def build_lora_config_for_spec(spec: Dict[str, Any], rank: int = LORA_RANK, alpha: Optional[int] = None):
    alpha = int(alpha if alpha is not None else LORA_ALPHA)
    lora_kwargs = dict(
        task_type=TaskType.CAUSAL_LM,
        r=int(rank),
        lora_alpha=alpha,
        lora_dropout=float(LORA_DROPOUT),
        bias="none",
        target_modules=spec.get("target_modules", []),
    )
    if spec.get("target_parameters"):
        sig = inspect.signature(LoraConfig)
        if "target_parameters" in sig.parameters:
            lora_kwargs["target_parameters"] = spec["target_parameters"]
            if not spec.get("target_modules"):
                lora_kwargs["target_modules"] = []
        else:
            print("⚠️ target_parameters not available in this PEFT build; ignoring experimental raw-parameter targeting.")
    return LoraConfig(**lora_kwargs)

def summarize_condition_specs(model_key: str, manifest: Dict[str, Any], specs: Dict[str, Any]) -> pd.DataFrame:
    total_params = sum(r["direct_param_count"] for r in manifest["records"])
    rows = []
    for condition_name, spec in specs.items():
        exact_targets = set(spec.get("target_modules", []))
        condition_params = 0
        for r in manifest["records"]:
            if r["full_name"] in exact_targets:
                condition_params += r["direct_param_count"]
        rows.append({
            "model_key": model_key,
            "condition": condition_name,
            "num_target_modules": len(spec.get("target_modules", [])),
            "num_target_parameters": len(spec.get("target_parameters", [])),
            "estimated_direct_params_in_targets": condition_params,
            "estimated_pct_of_model_direct_params": 100.0 * condition_params / max(total_params, 1),
            "notes": spec.get("notes", ""),
        })
    return pd.DataFrame(rows).sort_values(["model_key", "condition"]).reset_index(drop=True)


In [ ]:
# SECTION 1.3 — Verification: ensure LoRA targets are exactly what we think they are
# What this cell does:
# - instantiates a PEFT model for each condition
# - prints and records all trainable LoRA host modules
# - asserts that Falcon-H1 forbidden modules are not trainable
# - runs a quick forward pass
#
# Expected output:
# - verification summaries and trainable parameter counts

def strip_peft_prefix(name: str) -> str:
    prefixes = ["base_model.model.", "base_model."]
    for p in prefixes:
        if name.startswith(p):
            return name[len(p):]
    return name

def extract_lora_host_module_from_param_name(param_name: str) -> Optional[str]:
    name = strip_peft_prefix(param_name)
    for token in [".lora_A.", ".lora_B.", ".lora_embedding_A.", ".lora_embedding_B."]:
        if token in name:
            return name.split(token)[0]
    return None

def trainable_parameter_summary(model) -> Dict[str, Any]:
    total = 0
    trainable = 0
    names = []
    host_modules = []
    for name, p in model.named_parameters():
        n = p.numel()
        total += n
        if p.requires_grad:
            trainable += n
            names.append(name)
            host = extract_lora_host_module_from_param_name(name)
            if host is not None:
                host_modules.append(host)
    return {
        "trainable_params": trainable,
        "total_params": total,
        "pct_trainable": 100.0 * trainable / max(total, 1),
        "trainable_param_names": names,
        "lora_host_modules": sorted(set(host_modules)),
    }

def verify_condition(model_key: str, condition_name: str, spec: Dict[str, Any]) -> Dict[str, Any]:
    model, tokenizer = load_model_and_tokenizer(model_key, training=False)
    lora_config = build_lora_config_for_spec(spec)
    peft_model = get_peft_model(model, lora_config)
    peft_model.eval()

    summary = trainable_parameter_summary(peft_model)
    expected_hosts = set(spec.get("target_modules", []))
    actual_hosts = set(summary["lora_host_modules"])

    missing = sorted(expected_hosts - actual_hosts)
    unexpected = sorted(actual_hosts - expected_hosts)

    forbidden_hits = [h for h in actual_hosts if h.split(".")[-1] in FORBIDDEN_FALCON_SHORT_NAMES]
    assert not forbidden_hits, f"Forbidden Falcon-H1 modules received LoRA: {forbidden_hits}"

    test_text = "Hello! Please briefly explain what LoRA does."
    inputs = tokenizer(test_text, return_tensors="pt")
    if torch.cuda.is_available():
        peft_model = peft_model.cuda()
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        outputs = peft_model(**inputs)
        logits_shape = tuple(outputs.logits.shape)

    result = {
        "status": "complete",
        "model_key": model_key,
        "condition": condition_name,
        "expected_hosts": sorted(expected_hosts),
        "actual_hosts": sorted(actual_hosts),
        "missing_hosts": missing,
        "unexpected_hosts": unexpected,
        "trainable_params": summary["trainable_params"],
        "total_params": summary["total_params"],
        "pct_trainable": summary["pct_trainable"],
        "trainable_param_names": summary["trainable_param_names"],
        "logits_shape": logits_shape,
        "num_expected_hosts": len(expected_hosts),
        "num_actual_hosts": len(actual_hosts),
    }

    ckpt.save(f"verify__{model_key}__{condition_name}", result)
    del peft_model, model, tokenizer
    clear_memory()
    return result

def run_discovery_and_verification(models_to_run: List[str]) -> Dict[str, Any]:
    all_outputs = {}
    for model_key in models_to_run:
        print("=" * 120)
        print(f"DISCOVERY FOR {model_key}")
        manifest = discover_model_structure(model_key, force=False)
        specs = build_condition_specs(manifest)
        spec_df = summarize_condition_specs(model_key, manifest, specs)
        spec_df.to_csv(TABLES_DIR / f"{model_key}__condition_param_summary.csv", index=False)

        verification_rows = []
        for condition_name, spec in specs.items():
            print("-" * 120)
            print(f"Verifying {model_key} :: {condition_name}")
            try:
                result = verify_condition(model_key, condition_name, spec)
                verification_rows.append({
                    "model_key": model_key,
                    "condition": condition_name,
                    "trainable_params": result["trainable_params"],
                    "pct_trainable": result["pct_trainable"],
                    "num_expected_hosts": result["num_expected_hosts"],
                    "num_actual_hosts": result["num_actual_hosts"],
                    "missing_hosts": len(result["missing_hosts"]),
                    "unexpected_hosts": len(result["unexpected_hosts"]),
                    "status": result["status"],
                })
                print(f"✅ {condition_name} :: trainable={result['trainable_params']:,} "
                      f"({result['pct_trainable']:.4f}%) :: missing={len(result['missing_hosts'])} "
                      f":: unexpected={len(result['unexpected_hosts'])}")
            except Exception as e:
                traceback.print_exc()
                fail = {
                    "model_key": model_key,
                    "condition": condition_name,
                    "status": "failed",
                    "error": repr(e),
                    "traceback": traceback.format_exc(),
                }
                ckpt.save(f"verify__{model_key}__{condition_name}", fail)
                verification_rows.append({
                    "model_key": model_key,
                    "condition": condition_name,
                    "trainable_params": np.nan,
                    "pct_trainable": np.nan,
                    "num_expected_hosts": np.nan,
                    "num_actual_hosts": np.nan,
                    "missing_hosts": np.nan,
                    "unexpected_hosts": np.nan,
                    "status": "failed",
                })

        verification_df = pd.DataFrame(verification_rows)
        verification_df.to_csv(TABLES_DIR / f"{model_key}__verification_summary.csv", index=False)
        all_outputs[model_key] = {
            "manifest": manifest,
            "specs": specs,
            "spec_df": spec_df,
            "verification_df": verification_df,
        }
    return all_outputs


In [ ]:
# SECTION 1.4 — Execute discovery + verification and save summary figures
# What this cell does:
# - runs discovery and verification for each model
# - builds parameter-distribution figures
# - writes CSV tables and PNG/PDF figures to Drive
#
# Expected output:
# - printed tables
# - saved figures in /figures
# - NOTE reminding you to inspect outputs before training

discovery_outputs = run_discovery_and_verification(MODELS_TO_RUN)

def plot_component_pie(manifest: Dict[str, Any], out_prefix: Path):
    component_counts = defaultdict(int)
    for r in manifest["records"]:
        component_counts[r["component"]] += int(r["direct_param_count"])
    labels = []
    values = []
    for k, v in sorted(component_counts.items(), key=lambda kv: kv[1], reverse=True):
        if v <= 0:
            continue
        labels.append(k)
        values.append(v)

    plt.figure(figsize=(8, 8))
    plt.pie(values, labels=labels, autopct="%1.1f%%", startangle=90)
    plt.title(f"Parameter distribution by component — {manifest['model_key']}")
    plt.tight_layout()
    plt.savefig(str(out_prefix) + ".png", dpi=300, bbox_inches="tight")
    plt.savefig(str(out_prefix) + ".pdf", bbox_inches="tight")
    plt.close()

def plot_condition_bar(spec_df: pd.DataFrame, verification_df: pd.DataFrame, out_prefix: Path):
    merged = spec_df.merge(
        verification_df[["condition", "trainable_params", "pct_trainable", "status"]],
        on="condition",
        how="left",
    ).sort_values("trainable_params", ascending=False)

    plt.figure(figsize=(12, 5))
    plt.bar(merged["condition"], merged["trainable_params"].fillna(0))
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Trainable parameters")
    plt.title(f"Trainable LoRA parameters by condition — {merged['model_key'].iloc[0]}")
    plt.tight_layout()
    plt.savefig(str(out_prefix) + ".png", dpi=300, bbox_inches="tight")
    plt.savefig(str(out_prefix) + ".pdf", bbox_inches="tight")
    plt.close()

for model_key, out in discovery_outputs.items():
    manifest = out["manifest"]
    spec_df = out["spec_df"]
    verification_df = out["verification_df"]

    print(f"\n### {model_key} — condition summary")
    display(spec_df)

    print(f"\n### {model_key} — verification summary")
    display(verification_df)

    plot_component_pie(manifest, FIGURES_DIR / f"{model_key}__component_param_pie")
    plot_condition_bar(spec_df, verification_df, FIGURES_DIR / f"{model_key}__trainable_params_bar")

print("\n✅ Section 1 complete.")
print("MANUAL INSPECTION CHECKLIST:")
print("1) Open the saved module trees in /module_trees")
print("2) Inspect each condition summary CSV in /tables")
print("3) Confirm Falcon-H1 never targets conv1d or out_proj")
print("4) Only after that, set AUTO_RUN_EXPERIMENT_GRID = True")


# Section 2 — Dataset preparation and Drive caching

This section:
- downloads each training dataset once,
- formats it into model-appropriate training text,
- caches it to Drive,
- reserves fixed evaluation shot pools where needed.

The code is designed to be robust to minor dataset schema differences and to use fallbacks where practical.


In [ ]:
# SECTION 2.1 — Dataset registry, chat formatting, and cached preprocessing
# What this cell does:
# - defines dataset sources and fallback chains
# - formats examples into training text
# - caches processed DatasetDict objects to Drive
#
# Expected output:
# - function definitions only

TRAIN_DATASET_REGISTRY = {
    "ultrachat": {
        "candidates": [
            {"path": "HuggingFaceH4/ultrachat_200k", "name": None, "split": "train_sft"},
            {"path": "HuggingFaceH4/ultrachat_200k", "name": None, "split": "train"},
            {"path": "Open-Orca/OpenOrca", "name": None, "split": "train"},
        ],
        "type": "general",
    },
    "gsm8k_train": {
        "candidates": [
            {"path": "openai/gsm8k", "name": "main", "split": "train"},
        ],
        "type": "math",
    },
    "codealpaca": {
        "candidates": [
            {"path": "sahil2801/CodeAlpaca-20k", "name": None, "split": "train"},
            {"path": "iamtarun/python_code_instructions_18k_alpaca", "name": None, "split": "train"},
        ],
        "type": "code",
    },
}

def try_load_dataset_from_candidates(candidates: List[Dict[str, Any]]):
    last_err = None
    for cand in candidates:
        try:
            ds = load_dataset(cand["path"], cand.get("name"), split=cand["split"])
            print(f"Loaded dataset candidate: {cand}")
            return ds, cand
        except Exception as e:
            print(f"Failed candidate {cand}: {repr(e)}")
            last_err = e
    raise RuntimeError(f"All dataset candidates failed. Last error: {repr(last_err)}")

def fallback_chat_template(messages: List[Dict[str, str]], model_key: str, tokenizer=None, add_generation_prompt: bool = False) -> str:
    family = MODEL_REGISTRY[model_key]["family"]
    if family == "qwen3_5":
        chunks = []
        for m in messages:
            role = m["role"]
            content = m["content"].strip()
            chunks.append(f"{role.upper()}:\n{content}")
        if add_generation_prompt:
            chunks.append("ASSISTANT:\n")
        text = "\n\n".join(chunks)
        if tokenizer is not None and tokenizer.eos_token and not text.endswith(tokenizer.eos_token):
            text = text + tokenizer.eos_token
        return text

    chunks = []
    for m in messages:
        role = m["role"]
        content = m["content"].strip()
        chunks.append(f"{role.upper()}:\n{content}")
    if add_generation_prompt:
        chunks.append("ASSISTANT:\n")
    text = "\n\n".join(chunks)
    if tokenizer is not None and tokenizer.eos_token and not text.endswith(tokenizer.eos_token):
        text = text + tokenizer.eos_token
    return text

def safe_apply_chat_template(tokenizer, messages: List[Dict[str, str]], model_key: str, add_generation_prompt: bool = False) -> str:
    try:
        if getattr(tokenizer, "chat_template", None):
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=add_generation_prompt,
            )
    except Exception as e:
        print("⚠️ tokenizer.apply_chat_template failed; falling back:", repr(e))
    return fallback_chat_template(messages, model_key, tokenizer=tokenizer, add_generation_prompt=add_generation_prompt)

def format_ultrachat_example(example: Dict[str, Any], tokenizer, model_key: str) -> Dict[str, str]:
    if "messages" in example and example["messages"]:
        msgs = example["messages"]
        messages = [{"role": m["role"], "content": m["content"]} for m in msgs]
    else:
        system_prompt = example.get("system_prompt") or "You are a helpful assistant."
        question = example.get("question") or example.get("instruction") or ""
        response = example.get("response") or example.get("output") or ""
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
            {"role": "assistant", "content": response},
        ]
    text = safe_apply_chat_template(tokenizer, messages, model_key, add_generation_prompt=False)
    return {"text": text}

def format_gsm8k_example(example: Dict[str, Any], tokenizer, model_key: str) -> Dict[str, str]:
    question = example["question"].strip()
    answer = example["answer"].strip()
    messages = [
        {"role": "system", "content": "You are a careful mathematician. Solve step by step and finish with '#### <final answer>'."},
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    text = safe_apply_chat_template(tokenizer, messages, model_key, add_generation_prompt=False)
    return {"text": text}

def format_codealpaca_example(example: Dict[str, Any], tokenizer, model_key: str) -> Dict[str, str]:
    instruction = (example.get("instruction") or "").strip()
    inp = (example.get("input") or "").strip()
    output = (example.get("output") or "").strip()

    user_content = instruction
    if inp:
        user_content += "\n\nAdditional context:\n" + inp

    messages = [
        {"role": "system", "content": "You are a precise coding assistant. Write correct, readable code and explain only when useful."},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output},
    ]
    text = safe_apply_chat_template(tokenizer, messages, model_key, add_generation_prompt=False)
    return {"text": text}

def select_formatter(dataset_key: str):
    if dataset_key == "ultrachat":
        return format_ultrachat_example
    if dataset_key == "gsm8k_train":
        return format_gsm8k_example
    if dataset_key == "codealpaca":
        return format_codealpaca_example
    raise ValueError(f"Unknown dataset key: {dataset_key}")

def compute_text_length_stats(dataset: Dataset, tokenizer, sample_size: int = 256) -> Dict[str, Any]:
    n = min(sample_size, len(dataset))
    if n == 0:
        return {"count": 0}
    sample = dataset.select(range(n))
    lengths = []
    for row in sample:
        ids = tokenizer(row["text"], add_special_tokens=False)["input_ids"]
        lengths.append(len(ids))
    return {
        "count": len(lengths),
        "mean_tokens": float(np.mean(lengths)),
        "median_tokens": float(np.median(lengths)),
        "p90_tokens": float(np.percentile(lengths, 90)),
        "max_tokens": int(np.max(lengths)),
    }

def dataset_cache_path(dataset_key: str, model_key: str) -> Path:
    return DATA_CACHE_DIR / f"{dataset_key}__{model_key}"

def shot_pool_cache_path(name: str) -> Path:
    return DATA_CACHE_DIR / f"shot_pool__{name}"

def prepare_training_dataset(dataset_key: str, model_key: str, force: bool = False) -> DatasetDict:
    cache_path = dataset_cache_path(dataset_key, model_key)
    stats_path = DATA_CACHE_DIR / f"{dataset_key}__{model_key}__stats.json"

    if cache_path.exists() and not force:
        print(f"✅ Loading cached processed dataset: {cache_path}")
        return load_from_disk(str(cache_path))

    tokenizer = load_tokenizer(model_key)
    raw_ds, chosen_source = try_load_dataset_from_candidates(TRAIN_DATASET_REGISTRY[dataset_key]["candidates"])
    formatter = select_formatter(dataset_key)

    if dataset_key == "gsm8k_train":
        raw_ds = raw_ds.shuffle(seed=SEED)
        shot_pool_size = max(64, FEWSHOT_COUNTS["gsm8k"] * 16)
        shot_pool = raw_ds.select(range(min(shot_pool_size, len(raw_ds))))
        remaining = raw_ds.select(range(min(shot_pool_size, len(raw_ds)), len(raw_ds)))
        if not shot_pool_cache_path("gsm8k").exists():
            shot_pool.save_to_disk(str(shot_pool_cache_path("gsm8k")))
        raw_ds = remaining

    limit = min(MAX_TRAIN_SAMPLES_BY_DATASET[dataset_key], len(raw_ds))
    raw_ds = raw_ds.shuffle(seed=SEED).select(range(limit))

    processed = raw_ds.map(
        lambda ex: formatter(ex, tokenizer, model_key),
        remove_columns=raw_ds.column_names,
        desc=f"Formatting {dataset_key} for {model_key}",
    )

    test_size = min(max(64, int(0.05 * len(processed))), 256)
    if len(processed) <= test_size + 1:
        test_size = max(1, len(processed) // 10)

    dsd = processed.train_test_split(test_size=test_size, seed=SEED)
    dsd = DatasetDict({"train": dsd["train"], "validation": dsd["test"]})

    if cache_path.exists():
        shutil.rmtree(cache_path)
    dsd.save_to_disk(str(cache_path))

    stats = {
        "dataset_key": dataset_key,
        "model_key": model_key,
        "chosen_source": chosen_source,
        "train_size": len(dsd["train"]),
        "validation_size": len(dsd["validation"]),
        "text_length_stats_train": compute_text_length_stats(dsd["train"], tokenizer),
        "text_length_stats_validation": compute_text_length_stats(dsd["validation"], tokenizer),
    }
    save_text(stats_path, json.dumps(stats, indent=2, default=json_default))

    print(f"✅ Saved processed dataset to {cache_path}")
    print(json.dumps(stats, indent=2, default=json_default))
    return dsd


In [ ]:
# SECTION 2.2 — Prepare all training datasets for all active models
# What this cell does:
# - preprocesses every dataset × model pair and caches it to Drive
# - prints dataset statistics
#
# Expected output:
# - one cached DatasetDict per pair
# - JSON stats files in /data_cache

prepared_dataset_index = {}

for model_key in MODELS_TO_RUN:
    prepared_dataset_index[model_key] = {}
    for dataset_key in DATASETS_TO_USE:
        print("=" * 120)
        print(f"Preparing dataset {dataset_key} for model {model_key}")
        dsd = prepare_training_dataset(dataset_key, model_key, force=False)
        prepared_dataset_index[model_key][dataset_key] = {
            "train_size": len(dsd["train"]),
            "validation_size": len(dsd["validation"]),
            "path": str(dataset_cache_path(dataset_key, model_key)),
        }

prepared_dataset_index_path = DATA_CACHE_DIR / "prepared_dataset_index.json"
save_text(prepared_dataset_index_path, json.dumps(prepared_dataset_index, indent=2, default=json_default))
print("✅ Prepared dataset index saved to:", prepared_dataset_index_path)
print(json.dumps(prepared_dataset_index, indent=2, default=json_default))


# Sections 3 and 4 — Training and evaluation pipelines

The training loop is fully Drive-checkpointed and fault-tolerant:
- it skips completed experiments,
- resumes from the last Trainer checkpoint if available,
- saves adapters, logs, and evaluation details after every experiment.

The evaluation pipeline is **manual**:
- log-likelihood scoring for multiple-choice benchmarks,
- generation + regex extraction for GSM8K,
- optional HumanEval generation + code evaluation.


In [ ]:
# SECTION 3.1 / 4.1 — Benchmark loading, prompt construction, and manual evaluation helpers
# What this cell does:
# - loads MMLU / GSM8K / ARC-Challenge / HellaSwag / HumanEval with caching
# - defines few-shot prompt builders
# - implements manual log-likelihood scoring and generation-based evaluation
#
# Expected output:
# - function definitions only

BENCHMARK_REGISTRY = {
    "mmlu": {
        "loader": lambda: load_dataset("cais/mmlu", "all"),
        "eval_split": "validation",
        "shot_split": "dev",
    },
    "gsm8k": {
        "loader": lambda: load_dataset("openai/gsm8k", "main"),
        "eval_split": "test",
        "shot_split": None,
    },
    "arc_challenge": {
        "loader": lambda: load_dataset("ai2_arc", "ARC-Challenge"),
        "eval_split": "validation",
        "shot_split": "train",
    },
    "hellaswag": {
        "loader": lambda: load_dataset("Rowan/hellaswag"),
        "eval_split": "validation",
        "shot_split": "train",
    },
    "humaneval": {
        "loader": lambda: load_dataset("openai/openai_humaneval"),
        "eval_split": "test",
        "shot_split": None,
    },
}

def benchmark_cache_dir(benchmark_key: str) -> Path:
    return DATA_CACHE_DIR / f"benchmark__{benchmark_key}"

def ensure_benchmark_cached(benchmark_key: str):
    cache_dir = benchmark_cache_dir(benchmark_key)
    if cache_dir.exists():
        return load_from_disk(str(cache_dir))
    dsd = BENCHMARK_REGISTRY[benchmark_key]["loader"]()
    dsd.save_to_disk(str(cache_dir))
    return dsd

def get_benchmark_split(benchmark_key: str, split_name: str):
    dsd = ensure_benchmark_cached(benchmark_key)
    return dsd[split_name]

def get_fixed_eval_indices(benchmark_key: str, split_name: str, n_total: int, max_n: int) -> List[int]:
    idx_path = DATA_CACHE_DIR / f"indices__{benchmark_key}__{split_name}__{max_n}.json"
    if idx_path.exists():
        return json.loads(idx_path.read_text())
    rng = np.random.default_rng(SEED)
    if max_n >= n_total:
        indices = list(range(n_total))
    else:
        indices = sorted(rng.choice(np.arange(n_total), size=max_n, replace=False).tolist())
    save_text(idx_path, json.dumps(indices))
    return indices

def get_eval_subset(benchmark_key: str):
    split_name = BENCHMARK_REGISTRY[benchmark_key]["eval_split"]
    ds = get_benchmark_split(benchmark_key, split_name)
    max_n = min(MAX_EVAL_SAMPLES_BY_BENCHMARK[benchmark_key], len(ds))
    indices = get_fixed_eval_indices(benchmark_key, split_name, len(ds), max_n)
    return ds.select(indices)

def get_mc_choices(example: Dict[str, Any]) -> Tuple[str, List[str], int]:
    if "question" in example and "choices" in example and "answer" in example:
        question = example["question"]
        choices = example["choices"]
        gold = example["answer"]
        if isinstance(gold, str) and gold.isdigit():
            gold = int(gold)
        return question, list(choices), int(gold)

    if "question" in example and isinstance(example.get("choices"), dict):
        question = example["question"]
        texts = example["choices"]["text"]
        labels = example["choices"]["label"]
        gold_label = example["answerKey"]
        gold = labels.index(gold_label)
        return question, list(texts), int(gold)

    if "ctx" in example and "endings" in example:
        question = example["ctx"]
        choices = example["endings"]
        gold = int(example["label"])
        return question, list(choices), gold

    raise ValueError(f"Unsupported multiple-choice schema: keys={list(example.keys())}")

def choice_labels(n: int) -> List[str]:
    alpha = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    return [alpha[i] for i in range(n)]

def render_mc_example(example: Dict[str, Any], benchmark_key: str, include_answer: bool = False) -> str:
    q, choices, gold = get_mc_choices(example)
    labels = choice_labels(len(choices))
    lines = []

    if benchmark_key == "mmlu":
        subject = example.get("subject", "the subject")
        lines.append(f"The following is a multiple choice question about {subject}.")
        lines.append("")
        lines.append(q)
    elif benchmark_key == "arc_challenge":
        lines.append("Answer the following science question.")
        lines.append("")
        lines.append(q)
    elif benchmark_key == "hellaswag":
        lines.append("Choose the most plausible continuation.")
        lines.append("")
        lines.append(q)
    else:
        lines.append(q)

    for lab, ch in zip(labels, choices):
        lines.append(f"{lab}. {ch}")
    lines.append("Answer:")
    if include_answer:
        lines[-1] += f" {labels[gold]}"
    return "\n".join(lines).strip() + "\n\n"

def gsm8k_question(example: Dict[str, Any]) -> str:
    return example["question"].strip()

def gsm8k_answer(example: Dict[str, Any]) -> str:
    return example["answer"].strip()

def humaneval_prompt(example: Dict[str, Any]) -> str:
    return example["prompt"]

def render_gsm8k_example(example: Dict[str, Any], include_answer: bool = False) -> str:
    text = (
        "You are a careful mathematician. Solve the problem step by step and end with "
        "'#### <final answer>'.\n\n"
        f"Question: {gsm8k_question(example)}\nAnswer:"
    )
    if include_answer:
        text += " " + gsm8k_answer(example)
    return text.strip() + "\n\n"

def get_shot_pool(benchmark_key: str):
    if benchmark_key == "gsm8k":
        path = shot_pool_cache_path("gsm8k")
        if path.exists():
            return load_from_disk(str(path))
        ds = load_dataset("openai/gsm8k", "main", split="train")
        shot_pool = ds.shuffle(seed=SEED).select(range(min(len(ds), max(64, FEWSHOT_COUNTS["gsm8k"] * 16))))
        shot_pool.save_to_disk(str(path))
        return shot_pool

    split = BENCHMARK_REGISTRY[benchmark_key]["shot_split"]
    if split is None:
        return None
    return get_benchmark_split(benchmark_key, split)

def build_fewshot_prompt(tokenizer, benchmark_key: str, eval_example: Dict[str, Any], shot_pool, max_prompt_tokens: int = MAX_EVAL_PROMPT_TOKENS) -> str:
    shots_needed = FEWSHOT_COUNTS.get(benchmark_key, 0)
    shot_texts = []

    if shots_needed > 0 and shot_pool is not None:
        if benchmark_key == "mmlu":
            subject = eval_example.get("subject")
            same_subject = shot_pool.filter(lambda ex: ex.get("subject") == subject)
            source = same_subject if len(same_subject) >= shots_needed else shot_pool
            selected = source.select(range(min(shots_needed, len(source))))
            shot_texts = [render_mc_example(ex, benchmark_key, include_answer=True) for ex in selected]
            target = render_mc_example(eval_example, benchmark_key, include_answer=False)
        elif benchmark_key in {"arc_challenge", "hellaswag"}:
            selected = shot_pool.select(range(min(shots_needed, len(shot_pool))))
            shot_texts = [render_mc_example(ex, benchmark_key, include_answer=True) for ex in selected]
            target = render_mc_example(eval_example, benchmark_key, include_answer=False)
        elif benchmark_key == "gsm8k":
            selected = shot_pool.select(range(min(shots_needed, len(shot_pool))))
            shot_texts = [render_gsm8k_example(ex, include_answer=True) for ex in selected]
            target = render_gsm8k_example(eval_example, include_answer=False)
        else:
            target = ""
    else:
        if benchmark_key in {"mmlu", "arc_challenge", "hellaswag"}:
            target = render_mc_example(eval_example, benchmark_key, include_answer=False)
        elif benchmark_key == "gsm8k":
            target = render_gsm8k_example(eval_example, include_answer=False)
        else:
            target = ""

    chosen = list(shot_texts)
    while True:
        prompt = "".join(chosen + [target]).strip()
        toks = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        if len(toks) <= max_prompt_tokens or not chosen:
            return prompt
        chosen = chosen[1:]

def score_continuation_logprob(model, tokenizer, prompt: str, continuation: str) -> Dict[str, Any]:
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    full_ids = tokenizer(prompt + continuation, return_tensors="pt", add_special_tokens=False)
    device = next(model.parameters()).device
    input_ids = full_ids["input_ids"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids)
        logits = outputs.logits[:, :-1, :]
        targets = input_ids[:, 1:]
        prompt_len = prompt_ids["input_ids"].shape[1]
        cont_logits = logits[:, prompt_len - 1 :, :]
        cont_targets = targets[:, prompt_len - 1 :]
        log_probs = torch.log_softmax(cont_logits, dim=-1)
        token_log_probs = log_probs.gather(-1, cont_targets.unsqueeze(-1)).squeeze(-1)

    total_logprob = float(token_log_probs.sum().item())
    mean_logprob = float(token_log_probs.mean().item())
    return {
        "total_logprob": total_logprob,
        "mean_logprob": mean_logprob,
        "num_tokens": int(cont_targets.shape[1]),
    }

def extract_final_number(text: str) -> Optional[str]:
    m = re.findall(r"####\s*([-+]?[0-9][0-9,\.]*)", text)
    if m:
        return m[-1].replace(",", "").strip()
    m = re.findall(r"[-+]?[0-9][0-9,\.]*", text)
    if m:
        return m[-1].replace(",", "").strip()
    return None

def normalize_number_string(s: Optional[str]) -> Optional[str]:
    if s is None:
        return None
    s = s.strip().replace(",", "")
    try:
        val = float(s)
        if val.is_integer():
            return str(int(val))
        return str(val)
    except Exception:
        return s

def generate_text(model, tokenizer, prompt: str, max_new_tokens: int = 128) -> str:
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    gen = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    completion_ids = gen[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(completion_ids, skip_special_tokens=True)

def evaluate_multiple_choice(model, tokenizer, benchmark_key: str, label: str) -> Dict[str, Any]:
    assert benchmark_key in {"mmlu", "arc_challenge", "hellaswag"}
    eval_ds = get_eval_subset(benchmark_key)
    shot_pool = get_shot_pool(benchmark_key)

    detail_path = EVAL_DETAILS_DIR / f"{sanitize_key(label)}__{benchmark_key}.jsonl"
    if detail_path.exists():
        detail_path.unlink()

    correct = 0
    total = 0
    iterator = tqdm(eval_ds, total=len(eval_ds), desc=f"Eval {label} :: {benchmark_key}")
    for idx, ex in enumerate(iterator):
        prompt = build_fewshot_prompt(tokenizer, benchmark_key, ex, shot_pool)
        _, choices, gold = get_mc_choices(ex)
        labels = choice_labels(len(choices))
        scores = []
        for lab in labels:
            cont = " " + lab
            sc = score_continuation_logprob(model, tokenizer, prompt, cont)
            scores.append(sc["total_logprob"])
        pred = int(np.argmax(scores))
        ok = int(pred == gold)
        correct += ok
        total += 1
        row = {
            "benchmark": benchmark_key,
            "example_id": idx,
            "correct": ok,
            "pred_index": pred,
            "gold_index": gold,
            "pred_label": labels[pred],
            "gold_label": labels[gold],
            "score_vector": [float(x) for x in scores],
            "meta_subject": ex.get("subject"),
        }
        CheckpointManager.append_jsonl(detail_path, row)

    acc = correct / max(total, 1)
    return {"benchmark": benchmark_key, "label": label, "accuracy": acc, "n_eval": total, "detail_path": str(detail_path)}

def evaluate_gsm8k(model, tokenizer, label: str) -> Dict[str, Any]:
    benchmark_key = "gsm8k"
    eval_ds = get_eval_subset(benchmark_key)
    shot_pool = get_shot_pool(benchmark_key)

    detail_path = EVAL_DETAILS_DIR / f"{sanitize_key(label)}__{benchmark_key}.jsonl"
    if detail_path.exists():
        detail_path.unlink()

    correct = 0
    total = 0
    iterator = tqdm(eval_ds, total=len(eval_ds), desc=f"Eval {label} :: gsm8k")
    for idx, ex in enumerate(iterator):
        prompt = build_fewshot_prompt(tokenizer, benchmark_key, ex, shot_pool)
        completion = generate_text(model, tokenizer, prompt, max_new_tokens=GSM8K_MAX_NEW_TOKENS)
        pred_num = normalize_number_string(extract_final_number(completion))
        gold_num = normalize_number_string(extract_final_number(gsm8k_answer(ex)))
        ok = int(pred_num == gold_num)
        correct += ok
        total += 1
        row = {
            "benchmark": benchmark_key,
            "example_id": idx,
            "correct": ok,
            "pred_answer": pred_num,
            "gold_answer": gold_num,
            "raw_completion": completion,
        }
        CheckpointManager.append_jsonl(detail_path, row)

    acc = correct / max(total, 1)
    return {"benchmark": benchmark_key, "label": label, "accuracy": acc, "n_eval": total, "detail_path": str(detail_path)}

def evaluate_humaneval(model, tokenizer, label: str) -> Dict[str, Any]:
    benchmark_key = "humaneval"
    eval_ds = get_eval_subset(benchmark_key)

    detail_path = EVAL_DETAILS_DIR / f"{sanitize_key(label)}__{benchmark_key}.jsonl"
    if detail_path.exists():
        detail_path.unlink()

    predictions = []
    references = []
    iterator = tqdm(eval_ds, total=len(eval_ds), desc=f"Eval {label} :: humaneval")
    for idx, ex in enumerate(iterator):
        prompt = humaneval_prompt(ex)
        completion = generate_text(model, tokenizer, prompt, max_new_tokens=HUMANEVAL_MAX_NEW_TOKENS)
        predictions.append([completion])
        references.append(ex["test"])
        row = {"benchmark": benchmark_key, "example_id": idx, "task_id": ex["task_id"], "completion": completion}
        CheckpointManager.append_jsonl(detail_path, row)

    try:
        code_eval = evaluate.load("code_eval")
        score, _ = code_eval.compute(references=references, predictions=predictions, k=[1])
        pass_at_1 = float(score["pass@1"])
        metric_name = "pass@1"
    except Exception as e:
        print("⚠️ HumanEval code execution failed or code_eval metric unavailable:", repr(e))
        pass_at_1 = float("nan")
        metric_name = "pass@1_unavailable"

    return {
        "benchmark": benchmark_key,
        "label": label,
        "accuracy": pass_at_1,
        "metric_name": metric_name,
        "n_eval": len(eval_ds),
        "detail_path": str(detail_path),
    }

def evaluate_model_suite(model, tokenizer, model_key: str, label: str, train_dataset_key: Optional[str] = None) -> Dict[str, Any]:
    results = {}
    model.eval()
    results["mmlu"] = evaluate_multiple_choice(model, tokenizer, "mmlu", label)
    results["gsm8k"] = evaluate_gsm8k(model, tokenizer, label)
    results["arc_challenge"] = evaluate_multiple_choice(model, tokenizer, "arc_challenge", label)
    results["hellaswag"] = evaluate_multiple_choice(model, tokenizer, "hellaswag", label)

    should_run_humaneval = RUN_HUMANEVAL and (
        (not RUN_HUMANEVAL_ONLY_IF_CODE_TRAIN) or (train_dataset_key == "codealpaca")
    )
    if should_run_humaneval:
        results["humaneval"] = evaluate_humaneval(model, tokenizer, label)
    return results


In [ ]:
# SECTION 3.2 / 4.2 — Training callbacks, SFT config builder, baseline evaluation, and condition training
# What this cell does:
# - adds JSONL metric logging and LoRA-dynamics tracking
# - builds robust SFTConfig objects across TRL / Transformers versions
# - defines the reusable train_condition() function
#
# Expected output:
# - function definitions only

class JsonlLoggerCallback(TrainerCallback):
    def __init__(self, path: Path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        if self.path.exists():
            self.path.unlink()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        row = {"step": int(state.global_step), "epoch": float(state.epoch or 0.0), **logs}
        CheckpointManager.append_jsonl(self.path, row)

class LoraNormTrackerCallback(TrainerCallback):
    def __init__(self, path: Path, every_n_steps: int = LORA_DYNAMICS_EVERY_N_STEPS):
        self.path = Path(path)
        self.every_n_steps = every_n_steps
        self.path.parent.mkdir(parents=True, exist_ok=True)
        if self.path.exists():
            self.path.unlink()

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if model is None or state.global_step == 0:
            return
        if state.global_step % self.every_n_steps != 0:
            return

        rows = []
        for name, module in model.named_modules():
            if not hasattr(module, "lora_A") or not hasattr(module, "lora_B"):
                continue
            try:
                adapter_names = list(module.lora_A.keys())
            except Exception:
                adapter_names = ["default"]
            for adapter_name in adapter_names:
                try:
                    A = module.lora_A[adapter_name].weight.detach().float().cpu()
                    B = module.lora_B[adapter_name].weight.detach().float().cpu()
                    approx_fro = float(torch.norm(B @ A, p="fro").item())
                    rows.append({
                        "step": int(state.global_step),
                        "host_module": name,
                        "adapter_name": adapter_name,
                        "lora_A_fro": float(torch.norm(A, p="fro").item()),
                        "lora_B_fro": float(torch.norm(B, p="fro").item()),
                        "delta_fro_proxy": approx_fro,
                    })
                except Exception:
                    continue

        for row in rows:
            CheckpointManager.append_jsonl(self.path, row)

def build_sft_config(output_dir: str, eos_token: Optional[str] = None):
    common = dict(
        output_dir=output_dir,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        logging_steps=LOGGING_STEPS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=2,
        report_to=[],
        bf16=USE_BF16,
        fp16=USE_FP16,
        max_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        packing=False,
        remove_unused_columns=False,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
    )
    if eos_token is not None:
        common["eos_token"] = eos_token

    if EVAL_DURING_TRAINING:
        common["eval_steps"] = EVAL_EVERY_N_STEPS
        try:
            return SFTConfig(eval_strategy="steps", **common)
        except TypeError:
            return SFTConfig(evaluation_strategy="steps", **common)
    else:
        try:
            return SFTConfig(eval_strategy="no", **common)
        except TypeError:
            return SFTConfig(evaluation_strategy="no", **common)

def find_resume_checkpoint(output_dir: Path) -> Optional[str]:
    if not output_dir.exists():
        return None
    return get_last_checkpoint(str(output_dir))

def condition_specs_for_model(model_key: str) -> Dict[str, Any]:
    path = DISCOVERY_DIR / f"{model_key}__condition_specs.json"
    if path.exists():
        return json.loads(path.read_text())
    manifest = discover_model_structure(model_key)
    return build_condition_specs(manifest)

def summarize_trainable_parameters_live(model) -> Dict[str, Any]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        "total_params": int(total),
        "trainable_params": int(trainable),
        "pct_trainable": 100.0 * trainable / max(total, 1),
    }

def evaluate_base_model(model_key: str, force: bool = False) -> Dict[str, Any]:
    cache_key = f"base_eval__{model_key}"
    if not force:
        cached = ckpt.load(cache_key)
        if cached is not None and cached.get("status") == "complete":
            print(f"✅ Base evaluation already cached for {model_key}")
            return cached

    start = time.time()
    model, tokenizer = load_model_and_tokenizer(model_key, training=False)
    if torch.cuda.is_available():
        model = model.cuda()

    label = f"{model_key}__base"
    eval_summary = evaluate_model_suite(model, tokenizer, model_key, label, train_dataset_key=None)

    elapsed = time.time() - start
    result = {
        "status": "complete",
        "kind": "base_eval",
        "model_key": model_key,
        "condition": "__base__",
        "train_dataset": None,
        "seed": SEED,
        "rank": None,
        "eval_summary": eval_summary,
        "elapsed_seconds": elapsed,
        "elapsed_hours": elapsed / 3600.0,
        "estimated_cu": (elapsed / 3600.0) * ESTIMATED_CU_RATE if ESTIMATED_CU_RATE else None,
    }
    ckpt.save(cache_key, result)
    del model, tokenizer
    clear_memory()
    return result

def load_datasetdict_for_training(dataset_key: str, model_key: str) -> DatasetDict:
    path = dataset_cache_path(dataset_key, model_key)
    if not path.exists():
        prepare_training_dataset(dataset_key, model_key, force=False)
    return load_from_disk(str(path))

def train_condition(
    model_key: str,
    condition_name: str,
    train_dataset_key: str,
    seed: int = SEED,
    rank: int = LORA_RANK,
    lora_alpha: Optional[int] = None,
    experiment_suffix: str = "",
) -> Dict[str, Any]:
    exp_key = f"train__{model_key}__{condition_name}__{train_dataset_key}__seed{seed}{experiment_suffix}"
    cached = ckpt.load(exp_key)
    if cached is not None and cached.get("status") == "complete":
        print(f"✅ {exp_key} already completed.")
        return cached

    specs = condition_specs_for_model(model_key)
    if condition_name not in specs:
        raise KeyError(f"Condition {condition_name} not defined for {model_key}")
    spec = specs[condition_name]

    output_dir = MODELS_DIR / sanitize_key(exp_key)
    adapter_dir = output_dir / "final_adapter"
    log_path = LOGS_DIR / f"{sanitize_key(exp_key)}__train_log.jsonl"
    dynamics_path = LOGS_DIR / f"{sanitize_key(exp_key)}__lora_dynamics.jsonl"

    start = time.time()
    seed_everything(seed)

    try:
        dsd = load_datasetdict_for_training(train_dataset_key, model_key)
        train_ds = dsd["train"]
        val_ds = dsd["validation"]

        model, tokenizer = load_model_and_tokenizer(model_key, training=True)
        lora_cfg = build_lora_config_for_spec(spec, rank=rank, alpha=lora_alpha)
        model = get_peft_model(model, lora_cfg)

        live_summary = trainable_parameter_summary(model)
        expected_hosts = set(spec.get("target_modules", []))
        actual_hosts = set(live_summary["lora_host_modules"])
        unexpected = sorted(actual_hosts - expected_hosts)
        missing = sorted(expected_hosts - actual_hosts)
        if unexpected:
            print("⚠️ Unexpected trainable LoRA host modules:", unexpected[:20])
        if missing:
            print("⚠️ Missing expected LoRA host modules:", missing[:20])

        eos_token = tokenizer.eos_token if getattr(tokenizer, "eos_token", None) else None
        sft_args = build_sft_config(str(output_dir), eos_token=eos_token)
        callbacks = [JsonlLoggerCallback(log_path)]
        if ENABLE_LORA_DYNAMICS_TRACKING:
            callbacks.append(LoraNormTrackerCallback(dynamics_path))

        trainer = SFTTrainer(
            model=model,
            args=sft_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            processing_class=tokenizer,
            callbacks=callbacks,
        )

        resume_ckpt = find_resume_checkpoint(output_dir)
        if resume_ckpt:
            print("Resuming from checkpoint:", resume_ckpt)
            train_result = trainer.train(resume_from_checkpoint=resume_ckpt)
        else:
            train_result = trainer.train()

        trainer.save_model(str(adapter_dir))
        tokenizer.save_pretrained(str(adapter_dir))

        elapsed = time.time() - start
        train_summary = summarize_trainable_parameters_live(model)

        eval_summary = {}
        if RUN_INLINE_EVAL_AFTER_TRAIN:
            if torch.cuda.is_available():
                model = model.cuda()
            label = f"{exp_key}__inline_eval"
            eval_summary = evaluate_model_suite(model, tokenizer, model_key, label, train_dataset_key=train_dataset_key)

        result = {
            "status": "complete",
            "kind": "train_eval",
            "experiment_key": exp_key,
            "model_key": model_key,
            "condition": condition_name,
            "train_dataset": train_dataset_key,
            "seed": seed,
            "rank": rank,
            "lora_alpha": lora_alpha if lora_alpha is not None else LORA_ALPHA,
            "condition_spec_snapshot": spec,
            "train_summary": train_summary,
            "trainer_output_metrics": dict(getattr(train_result, "metrics", {})),
            "adapter_dir": str(adapter_dir),
            "output_dir": str(output_dir),
            "train_log_path": str(log_path),
            "lora_dynamics_path": str(dynamics_path) if ENABLE_LORA_DYNAMICS_TRACKING else None,
            "eval_summary": eval_summary,
            "elapsed_seconds": elapsed,
            "elapsed_hours": elapsed / 3600.0,
            "estimated_cu": (elapsed / 3600.0) * ESTIMATED_CU_RATE if ESTIMATED_CU_RATE else None,
            "num_train_examples": len(train_ds),
            "num_val_examples": len(val_ds),
        }

        ckpt.save(exp_key, result)
        return result

    except Exception as e:
        fail = {
            "status": "failed",
            "kind": "train_eval",
            "experiment_key": exp_key,
            "model_key": model_key,
            "condition": condition_name,
            "train_dataset": train_dataset_key,
            "seed": seed,
            "rank": rank,
            "lora_alpha": lora_alpha if lora_alpha is not None else LORA_ALPHA,
            "error": repr(e),
            "traceback": traceback.format_exc(),
        }
        ckpt.save(exp_key, fail)
        traceback.print_exc()
        return fail

    finally:
        clear_memory()

def load_adapter_for_eval(model_key: str, adapter_dir: str):
    model, tokenizer = load_model_and_tokenizer(model_key, training=False)
    peft_model = PeftModel.from_pretrained(model, adapter_dir, is_trainable=False)
    peft_model.eval()
    if torch.cuda.is_available():
        peft_model = peft_model.cuda()
    return peft_model, tokenizer

def evaluate_saved_adapter(model_key: str, adapter_dir: str, condition_name: str, train_dataset_key: str, seed: int = SEED):
    exp_key = f"adapter_eval__{model_key}__{condition_name}__{train_dataset_key}__seed{seed}"
    cached = ckpt.load(exp_key)
    if cached is not None and cached.get("status") == "complete":
        print(f"✅ {exp_key} already completed.")
        return cached

    try:
        model, tokenizer = load_adapter_for_eval(model_key, adapter_dir)
        label = f"{exp_key}__eval"
        eval_summary = evaluate_model_suite(model, tokenizer, model_key, label, train_dataset_key=train_dataset_key)
        result = {
            "status": "complete",
            "kind": "adapter_eval",
            "experiment_key": exp_key,
            "model_key": model_key,
            "condition": condition_name,
            "train_dataset": train_dataset_key,
            "seed": seed,
            "adapter_dir": adapter_dir,
            "eval_summary": eval_summary,
        }
        ckpt.save(exp_key, result)
        return result
    except Exception as e:
        fail = {
            "status": "failed",
            "kind": "adapter_eval",
            "experiment_key": exp_key,
            "model_key": model_key,
            "condition": condition_name,
            "train_dataset": train_dataset_key,
            "seed": seed,
            "adapter_dir": adapter_dir,
            "error": repr(e),
            "traceback": traceback.format_exc(),
        }
        ckpt.save(exp_key, fail)
        traceback.print_exc()
        return fail
    finally:
        clear_memory()


In [ ]:
# SECTION 3.3 / 4.3 — Experiment grid execution
# What this cell does:
# - optionally runs baseline evaluation
# - optionally launches the full model × condition × dataset grid
# - continues on failure instead of aborting the notebook
#
# Expected output:
# - baseline caches
# - per-experiment training/evaluation logs
# - skipped runs if already completed

def core_conditions_for_model(model_key: str) -> List[str]:
    return list(condition_specs_for_model(model_key).keys())

def reduced_rank_ablation_conditions(model_key: str) -> List[str]:
    family = MODEL_REGISTRY[model_key]["family"]
    if family == "qwen3_5":
        return ["all_layers", "softmax_only", "gdn_only"]
    if family == "falcon_h1":
        return ["all_eligible", "attention_only", "ssm_only"]
    return list(condition_specs_for_model(model_key).keys())[:3]

def run_experiment_grid():
    results = []

    if RUN_BASELINE_EVALS_FIRST:
        for model_key in MODELS_TO_RUN:
            print("=" * 120)
            print(f"Running / loading baseline evaluation for {model_key}")
            try:
                results.append(evaluate_base_model(model_key, force=False))
            except Exception:
                traceback.print_exc()

    for model_key in MODELS_TO_RUN:
        conditions = core_conditions_for_model(model_key)
        for dataset_key in DATASETS_TO_USE:
            for condition_name in conditions:
                print("=" * 120)
                print(f"TRAINING: model={model_key} | condition={condition_name} | dataset={dataset_key}")
                try:
                    result = train_condition(
                        model_key=model_key,
                        condition_name=condition_name,
                        train_dataset_key=dataset_key,
                        seed=SEED,
                        rank=LORA_RANK,
                        lora_alpha=LORA_ALPHA,
                        experiment_suffix="",
                    )
                    results.append(result)
                except Exception:
                    traceback.print_exc()
                    continue

    if ENABLE_RANK_ABLATION:
        for model_key in MODELS_TO_RUN:
            for dataset_key in DATASETS_TO_USE:
                for condition_name in reduced_rank_ablation_conditions(model_key):
                    for rank in RANK_ABLATION_RANKS:
                        print("=" * 120)
                        print(f"RANK ABLATION: model={model_key} | condition={condition_name} | dataset={dataset_key} | rank={rank}")
                        try:
                            result = train_condition(
                                model_key=model_key,
                                condition_name=condition_name,
                                train_dataset_key=dataset_key,
                                seed=SEED,
                                rank=rank,
                                lora_alpha=2 * rank,
                                experiment_suffix=f"__rank{rank}",
                            )
                            results.append(result)
                        except Exception:
                            traceback.print_exc()
                            continue

    if ENABLE_MULTI_SEED_CORE:
        for model_key in MODELS_TO_RUN:
            family = MODEL_REGISTRY[model_key]["family"]
            if family == "qwen3_5":
                seed_conditions = ["all_layers", "softmax_only", "gdn_only"]
            else:
                seed_conditions = ["all_eligible", "attention_only", "ssm_only"]

            for dataset_key in DATASETS_TO_USE:
                for condition_name in seed_conditions:
                    for seed in OPTIONAL_MULTI_SEEDS:
                        print("=" * 120)
                        print(f"MULTI-SEED: model={model_key} | condition={condition_name} | dataset={dataset_key} | seed={seed}")
                        try:
                            result = train_condition(
                                model_key=model_key,
                                condition_name=condition_name,
                                train_dataset_key=dataset_key,
                                seed=seed,
                                rank=LORA_RANK,
                                lora_alpha=LORA_ALPHA,
                                experiment_suffix=f"__seed{seed}",
                            )
                            results.append(result)
                        except Exception:
                            traceback.print_exc()
                            continue
    return results

# ── KILL SWITCH: Quick sanity check after first block ──────────────────
def run_kill_switch_block(model_key="qwen3_5_0_8b_base", dataset_key="gsm8k_train"):
    """Run one model × one domain × all conditions. Check if placement differences exist.
    
    🔴 Spread < 1%  → Paper lacks signal. Rethink.
    🟡 Spread 1-3%  → Borderline. Add multi-seed early.
    🟢 Spread 3-5%  → Good. Proceed.
    🟢🟢 Spread ≥ 5% → Strong. Full speed ahead.
    """
    conditions = core_conditions_for_model(model_key)
    results = []
    
    print("=" * 100)
    print(f"🔍 KILL SWITCH: {model_key} × {dataset_key} × {len(conditions)} conditions")
    print("=" * 100)
    
    try:
        evaluate_base_model(model_key, force=False)
    except Exception:
        traceback.print_exc()
    
    for cond in conditions:
        print(f"\n{'='*80}\n  Training: {cond}\n{'='*80}")
        try:
            results.append(train_condition(
                model_key=model_key, condition_name=cond,
                train_dataset_key=dataset_key, seed=SEED,
                rank=LORA_RANK, lora_alpha=LORA_ALPHA, experiment_suffix="",
            ))
        except Exception:
            traceback.print_exc()
    
    # ── Analyze spread ──
    target_bm = {"gsm8k_train":"gsm8k", "codealpaca":"humaneval", "ultrachat":"mmlu"}[dataset_key]
    scores = {}
    for r in results:
        if isinstance(r, dict) and r.get("status") == "complete":
            acc = (r.get("eval_summary") or {}).get(target_bm, {}).get("accuracy")
            if acc is not None:
                scores[r["condition"]] = acc
    
    print(f"\n{'='*100}")
    print(f"📊 KILL SWITCH RESULTS — {target_bm}")
    print(f"{'='*100}")
    for c, s in sorted(scores.items(), key=lambda x: -x[1]):
        print(f"  {c:28s} {s:.4f}")
    
    if len(scores) >= 3:
        spread = max(scores.values()) - min(scores.values())
        print(f"\n  SPREAD: {spread:.4f} ({spread*100:.1f}%)")
        if spread < 0.01:
            print("  🔴 < 1% — All conditions nearly identical. STOP and rethink.")
        elif spread < 0.03:
            print("  🟡 1-3% — Borderline. Consider multi-seed to confirm.")
        elif spread < 0.05:
            print("  🟢 3-5% — Good signal. Proceed with grid.")
        else:
            print("  🟢🟢 ≥ 5% — Strong signal. Full speed ahead!")
    
    return results


# ── Main execution block ──────────────────────────────────────────────
if AUTO_RUN_EXPERIMENT_GRID:
    experiment_outputs = run_experiment_grid()
    print(f"✅ Grid finished: {len(experiment_outputs)} outputs.")
else:
    experiment_outputs = []
    print("AUTO_RUN_EXPERIMENT_GRID = False")
    print()
    print("▶ STEP 1: Run kill switch first:")
    print("    kill_results = run_kill_switch_block()")
    print()
    print("▶ STEP 2: If green, run full grid:")
    print("    grid_results = run_experiment_grid()")


# Optional Section — Layer Card proxy, LoRA dynamics summaries, and extra diagnostics

These cells are optional but useful for analysis and discussion. They are included here because they materially strengthen the paper if compute permits.


In [ ]:
# OPTIONAL SECTION — Cheap Layer Card proxy and LoRA-dynamics summarizers
# What this cell does:
# - computes a low-cost per-layer / per-component gradient-sensitivity proxy
# - summarizes LoRA norm-growth logs into publication-ready tables
#
# Expected output:
# - CSV + figures under /analysis_cache, /tables, /figures

def infer_component_from_param_name(param_name: str, manifest: Dict[str, Any]) -> str:
    clean = strip_peft_prefix(param_name)
    module_name = clean.rsplit(".", 1)[0]
    exact = [r for r in manifest["records"] if r["full_name"] == module_name]
    if exact:
        return exact[0]["component"]
    for r in manifest["records"]:
        if module_name.startswith(r["full_name"] + "."):
            return r["component"]
    return "other"

def compute_layercard_proxy(model_key: str, calibration_dataset_key: str = "ultrachat", num_batches: int = 8):
    if not ENABLE_LAYERCARD_PROXY:
        print("Layer Card proxy is disabled.")
        return None

    out_csv = ANALYSIS_CACHE_DIR / f"{model_key}__layercard_proxy.csv"
    if out_csv.exists():
        df = pd.read_csv(out_csv)
        print(f"✅ Loaded cached Layer Card proxy from {out_csv}")
        return df

    manifest = ckpt.load(f"discovery__{model_key}")
    if manifest is None:
        manifest = discover_model_structure(model_key)

    model, tokenizer = load_model_and_tokenizer(model_key, training=False)
    model.train()
    if torch.cuda.is_available():
        model = model.cuda()

    dsd = load_datasetdict_for_training(calibration_dataset_key, model_key)
    subset = dsd["train"].select(range(min(num_batches, len(dsd["train"]))))

    accum = defaultdict(lambda: {"grad_norm_sum": 0.0, "param_count": 0})

    for row in tqdm(subset, desc=f"Layer Card proxy :: {model_key}"):
        batch = tokenizer(row["text"], truncation=True, max_length=MAX_SEQ_LENGTH, return_tensors="pt")
        if torch.cuda.is_available():
            batch = {k: v.cuda() for k, v in batch.items()}
        outputs = model(**batch, labels=batch["input_ids"])
        loss = outputs.loss
        loss.backward()

        for name, p in model.named_parameters():
            if p.grad is None:
                continue
            layer_idx = extract_layer_idx_from_name(name)
            if layer_idx is None:
                continue
            comp = infer_component_from_param_name(name, manifest)
            key = (layer_idx, comp)
            accum[key]["grad_norm_sum"] += float(torch.norm(p.grad.detach().float()).item())
            accum[key]["param_count"] += int(p.numel())

        model.zero_grad(set_to_none=True)

    rows = []
    for (layer_idx, comp), vals in accum.items():
        param_count = max(vals["param_count"], 1)
        rows.append({
            "model_key": model_key,
            "layer_idx": int(layer_idx),
            "component": comp,
            "grad_norm_sum": vals["grad_norm_sum"],
            "param_count": param_count,
            "score_per_million_params": vals["grad_norm_sum"] / (param_count / 1e6),
        })

    df = pd.DataFrame(rows).sort_values(["score_per_million_params"], ascending=False)
    df.to_csv(out_csv, index=False)

    plt.figure(figsize=(12, 5))
    show = df.head(30).copy()
    labels = [f"L{r.layer_idx}:{r.component}" for r in show.itertuples()]
    plt.bar(labels, show["score_per_million_params"])
    plt.xticks(rotation=70, ha="right")
    plt.ylabel("Proxy score (grad norm / million params)")
    plt.title(f"Layer Card proxy — top sensitive layer/components — {model_key}")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"{model_key}__layercard_proxy_top30.png", dpi=300, bbox_inches="tight")
    plt.savefig(FIGURES_DIR / f"{model_key}__layercard_proxy_top30.pdf", bbox_inches="tight")
    plt.close()

    del model, tokenizer
    clear_memory()
    return df

def summarize_lora_dynamics_logs():
    rows = []
    for path in LOGS_DIR.glob("*__lora_dynamics.jsonl"):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                row = json.loads(line)
                row["source_file"] = str(path)
                rows.append(row)
    if not rows:
        print("No LoRA dynamics logs found yet.")
        return None

    df = pd.DataFrame(rows)
    summary = (
        df.groupby(["source_file", "host_module"], as_index=False)
          .agg(
              max_delta_fro_proxy=("delta_fro_proxy", "max"),
              last_delta_fro_proxy=("delta_fro_proxy", "last"),
              max_step=("step", "max"),
          )
          .sort_values("last_delta_fro_proxy", ascending=False)
    )
    out_csv = TABLES_DIR / "lora_dynamics_summary.csv"
    summary.to_csv(out_csv, index=False)
    print("✅ Saved LoRA dynamics summary to:", out_csv)
    display(summary.head(30))
    return summary


In [ ]:
# OPTIONAL SECTION — Run the Layer Card proxy (cheap diagnostic) on active models
# What this cell does:
# - computes a proxy layer ranking for later comparison with the empirical winning conditions
#
# Expected output:
# - per-model CSVs and top-30 bar charts

if ENABLE_LAYERCARD_PROXY:
    layercard_outputs = {}
    for model_key in MODELS_TO_RUN:
        print("=" * 120)
        print(f"Layer Card proxy for {model_key}")
        try:
            layercard_outputs[model_key] = compute_layercard_proxy(model_key, calibration_dataset_key="ultrachat", num_batches=8)
            display(layercard_outputs[model_key].head(20))
        except Exception:
            traceback.print_exc()
else:
    layercard_outputs = {}
    print("Layer Card proxy is disabled.")


# Section 5 and 6 — Analysis, statistical comparison, figures, LaTeX tables, and summary JSON

This final section aggregates everything that was cached in Drive and exports:
- benchmark matrices
- delta-vs-base and delta-vs-full comparisons
- efficiency ratios
- forgetting summaries
- bootstrap comparisons for key pairs
- publication-ready figures
- LaTeX tables
- a machine-readable summary JSON


In [ ]:
# SECTION 5 / 6 — Aggregate checkpoints and build publication outputs
# What this cell does:
# - reads all checkpointed results
# - builds tidy result tables
# - computes deltas, efficiency, forgetting, and paired bootstrap comparisons
# - saves figures and LaTeX tables
#
# Expected output:
# - CSVs in /tables
# - PNG/PDF figures in /figures
# - summary JSON in /results/summary

TARGET_BENCHMARKS_BY_TRAIN_DATASET = {
    "ultrachat": {"mmlu", "arc_challenge", "hellaswag"},
    "gsm8k_train": {"gsm8k"},
    "codealpaca": {"humaneval"},
}

FULL_BASELINE_CONDITION_BY_FAMILY = {
    "qwen3_5": "all_layers",
    "falcon_h1": "all_eligible",
}

def load_all_checkpoint_payloads() -> List[Dict[str, Any]]:
    payloads = []
    for path in ckpt.list_paths(".pkl"):
        try:
            with open(path, "rb") as f:
                payloads.append(pickle.load(f))
        except Exception:
            print("⚠️ Failed to load checkpoint:", path)
    return payloads

def flatten_results(payloads: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for obj in payloads:
        if not isinstance(obj, dict):
            continue
        if obj.get("status") != "complete":
            continue
        eval_summary = obj.get("eval_summary", {})
        if not eval_summary:
            continue
        for benchmark, metrics in eval_summary.items():
            rows.append({
                "kind": obj.get("kind"),
                "experiment_key": obj.get("experiment_key"),
                "model_key": obj.get("model_key"),
                "condition": obj.get("condition"),
                "train_dataset": obj.get("train_dataset"),
                "seed": obj.get("seed"),
                "rank": obj.get("rank"),
                "lora_alpha": obj.get("lora_alpha"),
                "benchmark": benchmark,
                "accuracy": metrics.get("accuracy"),
                "n_eval": metrics.get("n_eval"),
                "detail_path": metrics.get("detail_path"),
                "metric_name": metrics.get("metric_name", "accuracy"),
                "elapsed_hours": obj.get("elapsed_hours"),
                "estimated_cu": obj.get("estimated_cu"),
                "adapter_dir": obj.get("adapter_dir"),
                "trainable_params": (obj.get("train_summary") or {}).get("trainable_params"),
                "pct_trainable": (obj.get("train_summary") or {}).get("pct_trainable"),
            })
    return pd.DataFrame(rows)

def family_for_model(model_key: str) -> str:
    return MODEL_REGISTRY[model_key]["family"]

def add_baseline_deltas(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    base_df = out[out["condition"] == "__base__"][["model_key", "benchmark", "accuracy"]].rename(columns={"accuracy": "base_accuracy"})
    out = out.merge(base_df, on=["model_key", "benchmark"], how="left")
    out["delta_vs_base"] = out["accuracy"] - out["base_accuracy"]

    full_rows = []
    for (model_key, train_dataset, benchmark, seed, rank), sub in out.groupby(["model_key", "train_dataset", "benchmark", "seed", "rank"], dropna=False):
        fam = family_for_model(model_key)
        full_condition = FULL_BASELINE_CONDITION_BY_FAMILY[fam]
        full_sub = sub[sub["condition"] == full_condition]
        full_acc = full_sub["accuracy"].iloc[0] if len(full_sub) else np.nan
        for idx in sub.index:
            full_rows.append((idx, full_acc))

    full_acc_series = pd.Series({idx: acc for idx, acc in full_rows}, name="full_condition_accuracy")
    out = out.join(full_acc_series, how="left")
    out["delta_vs_full"] = out["accuracy"] - out["full_condition_accuracy"]
    out["efficiency_ratio"] = out["delta_vs_base"] / out["estimated_cu"].replace({0: np.nan})
    return out

def compute_forgetting_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    train_df = df[df["condition"].notna() & (df["condition"] != "__base__")].copy()
    for keys, sub in train_df.groupby(["model_key", "condition", "train_dataset", "seed", "rank"], dropna=False):
        model_key, condition, train_dataset, seed, rank = keys
        target_benchmarks = TARGET_BENCHMARKS_BY_TRAIN_DATASET.get(train_dataset, set())
        non_target = sub[~sub["benchmark"].isin(target_benchmarks)]
        target = sub[sub["benchmark"].isin(target_benchmarks)]
        rows.append({
            "model_key": model_key,
            "condition": condition,
            "train_dataset": train_dataset,
            "seed": seed,
            "rank": rank,
            "target_mean_delta_vs_base": float(target["delta_vs_base"].mean()) if len(target) else np.nan,
            "non_target_mean_delta_vs_base": float(non_target["delta_vs_base"].mean()) if len(non_target) else np.nan,
            "forgetting_score": float(-non_target["delta_vs_base"].mean()) if len(non_target) else np.nan,
            "mean_accuracy": float(sub["accuracy"].mean()) if len(sub) else np.nan,
            "mean_efficiency_ratio": float(sub["efficiency_ratio"].mean()) if len(sub) else np.nan,
            "trainable_params": float(sub["trainable_params"].dropna().iloc[0]) if sub["trainable_params"].notna().any() else np.nan,
            "pct_trainable": float(sub["pct_trainable"].dropna().iloc[0]) if sub["pct_trainable"].notna().any() else np.nan,
            "estimated_cu": float(sub["estimated_cu"].dropna().iloc[0]) if sub["estimated_cu"].notna().any() else np.nan,
        })
    return pd.DataFrame(rows)

def read_detail_correctness(detail_path: str) -> Dict[Any, int]:
    data = {}
    if detail_path is None or (not Path(detail_path).exists()):
        return data
    with open(detail_path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            data[row["example_id"]] = int(row["correct"])
    return data

def paired_bootstrap(detail_path_a: str, detail_path_b: str, n_boot: int = 2000, seed: int = SEED) -> Dict[str, Any]:
    a = read_detail_correctness(detail_path_a)
    b = read_detail_correctness(detail_path_b)
    shared = sorted(set(a) & set(b))
    if not shared:
        return {"n_shared": 0, "mean_diff": np.nan, "ci_low": np.nan, "ci_high": np.nan}

    arr_a = np.array([a[i] for i in shared], dtype=np.float32)
    arr_b = np.array([b[i] for i in shared], dtype=np.float32)
    diffs = arr_a - arr_b

    rng = np.random.default_rng(seed)
    boot = []
    n = len(diffs)
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot.append(float(diffs[idx].mean()))
    boot = np.array(boot)
    return {
        "n_shared": int(n),
        "mean_diff": float(diffs.mean()),
        "ci_low": float(np.percentile(boot, 2.5)),
        "ci_high": float(np.percentile(boot, 97.5)),
    }

def save_table(df: pd.DataFrame, stem: str):
    csv_path = TABLES_DIR / f"{stem}.csv"
    tex_path = TABLES_DIR / f"{stem}.tex"
    df.to_csv(csv_path, index=False)
    try:
        with open(tex_path, "w", encoding="utf-8") as f:
            f.write(df.to_latex(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
    except Exception as e:
        print("⚠️ Failed to write LaTeX table:", stem, repr(e))
    return csv_path, tex_path

def make_grouped_bar(df: pd.DataFrame, model_key: str, out_prefix: Path):
    sub = df[(df["model_key"] == model_key) & (df["condition"] != "__base__")].copy()
    if sub.empty:
        return
    plot_df = (
        sub.groupby(["condition"], as_index=False)["delta_vs_base"]
           .mean()
           .sort_values("delta_vs_base", ascending=False)
    )
    plt.figure(figsize=(12, 5))
    plt.bar(plot_df["condition"], plot_df["delta_vs_base"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Mean delta vs base")
    plt.title(f"Mean benchmark gain by condition — {model_key}")
    plt.tight_layout()
    plt.savefig(str(out_prefix) + ".png", dpi=300, bbox_inches="tight")
    plt.savefig(str(out_prefix) + ".pdf", bbox_inches="tight")
    plt.close()

def make_heatmap(df: pd.DataFrame, model_key: str, train_dataset: str, out_prefix: Path):
    sub = df[(df["model_key"] == model_key) & (df["train_dataset"] == train_dataset) & (df["condition"] != "__base__")].copy()
    if sub.empty:
        return
    pivot = sub.pivot_table(index="benchmark", columns="condition", values="delta_vs_base", aggfunc="mean")
    plt.figure(figsize=(12, 5))
    plt.imshow(pivot.values, aspect="auto")
    plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=45, ha="right")
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.colorbar(label="Delta vs base")
    plt.title(f"Benchmark × condition heatmap — {model_key} — trained on {train_dataset}")
    plt.tight_layout()
    plt.savefig(str(out_prefix) + ".png", dpi=300, bbox_inches="tight")
    plt.savefig(str(out_prefix) + ".pdf", bbox_inches="tight")
    plt.close()

def make_scatter_pareto(forgetting_df: pd.DataFrame, model_key: str, out_prefix: Path):
    sub = forgetting_df[forgetting_df["model_key"] == model_key].copy()
    if sub.empty:
        return
    plt.figure(figsize=(8, 6))
    plt.scatter(sub["trainable_params"], sub["mean_accuracy"])
    for r in sub.itertuples():
        plt.annotate(r.condition, (r.trainable_params, r.mean_accuracy), fontsize=8)
    plt.xlabel("Trainable parameters")
    plt.ylabel("Mean benchmark accuracy")
    plt.title(f"Params vs performance — {model_key}")
    plt.tight_layout()
    plt.savefig(str(out_prefix) + ".png", dpi=300, bbox_inches="tight")
    plt.savefig(str(out_prefix) + ".pdf", bbox_inches="tight")
    plt.close()

def parse_train_log(path: str) -> pd.DataFrame:
    rows = []
    if path is None or not Path(path).exists():
        return pd.DataFrame()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if "loss" in row:
                rows.append(row)
    return pd.DataFrame(rows)

def make_loss_overlay(payloads: List[Dict[str, Any]], model_key: str, train_dataset: str, out_prefix: Path):
    plt.figure(figsize=(12, 5))
    drew_any = False
    for obj in payloads:
        if obj.get("status") != "complete":
            continue
        if obj.get("kind") != "train_eval":
            continue
        if obj.get("model_key") != model_key or obj.get("train_dataset") != train_dataset:
            continue
        log_df = parse_train_log(obj.get("train_log_path"))
        if log_df.empty:
            continue
        plt.plot(log_df["step"], log_df["loss"], label=obj["condition"])
        drew_any = True
    if not drew_any:
        plt.close()
        return
    plt.xlabel("Step")
    plt.ylabel("Training loss")
    plt.title(f"Training loss curves — {model_key} — trained on {train_dataset}")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(str(out_prefix) + ".png", dpi=300, bbox_inches="tight")
    plt.savefig(str(out_prefix) + ".pdf", bbox_inches="tight")
    plt.close()

def make_radar(df: pd.DataFrame, model_key: str, train_dataset: str, conditions: List[str], out_prefix: Path):
    sub = df[(df["model_key"] == model_key) & (df["train_dataset"] == train_dataset)]
    if sub.empty:
        return
    benchs = sorted(sub["benchmark"].unique().tolist())
    angles = np.linspace(0, 2 * np.pi, len(benchs), endpoint=False).tolist()
    angles += angles[:1]

    plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, polar=True)

    for condition in conditions:
        cur = sub[sub["condition"] == condition]
        if cur.empty:
            continue
        vals = []
        for b in benchs:
            row = cur[cur["benchmark"] == b]
            vals.append(float(row["accuracy"].mean()) if len(row) else np.nan)
        vals += vals[:1]
        ax.plot(angles, vals, label=condition)
        ax.fill(angles, vals, alpha=0.05)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(benchs)
    ax.set_title(f"Benchmark radar — {model_key} — trained on {train_dataset}")
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.15))
    plt.tight_layout()
    plt.savefig(str(out_prefix) + ".png", dpi=300, bbox_inches="tight")
    plt.savefig(str(out_prefix) + ".pdf", bbox_inches="tight")
    plt.close()

def derive_practitioner_recipe(forgetting_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (model_key, train_dataset), sub in forgetting_df.groupby(["model_key", "train_dataset"]):
        if sub.empty:
            continue
        full_condition = FULL_BASELINE_CONDITION_BY_FAMILY[family_for_model(model_key)]
        full_row = sub[sub["condition"] == full_condition]
        if len(full_row):
            full_acc = float(full_row["mean_accuracy"].iloc[0])
        else:
            full_acc = float(sub["mean_accuracy"].max())
        eligible = sub[sub["mean_accuracy"] >= 0.95 * full_acc].copy()
        if eligible.empty:
            eligible = sub.copy()
        chosen = eligible.sort_values(["trainable_params", "estimated_cu", "forgetting_score"], ascending=[True, True, True]).iloc[0]
        rows.append({
            "model_key": model_key,
            "train_dataset": train_dataset,
            "recommended_condition": chosen["condition"],
            "recommended_mean_accuracy": chosen["mean_accuracy"],
            "recommended_trainable_params": chosen["trainable_params"],
            "recommended_estimated_cu": chosen["estimated_cu"],
            "full_reference_accuracy": full_acc,
            "within_95pct_of_full": bool(chosen["mean_accuracy"] >= 0.95 * full_acc),
            "reason": "Smallest trainable-parameter / CU configuration within 95% of the full baseline mean accuracy.",
        })
    return pd.DataFrame(rows)

all_payloads = load_all_checkpoint_payloads()
flat_df = flatten_results(all_payloads)

if flat_df.empty:
    print("No completed evaluation payloads found yet. Run training/evaluation first.")
else:
    flat_df = add_baseline_deltas(flat_df)
    forgetting_df = compute_forgetting_summary(flat_df)

    save_table(flat_df, "all_results_flat")
    save_table(forgetting_df, "forgetting_summary")

    result_matrix = (
        flat_df[flat_df["condition"] != "__base__"]
        .pivot_table(index=["model_key", "train_dataset", "condition"], columns="benchmark", values="accuracy", aggfunc="mean")
        .reset_index()
    )
    save_table(result_matrix, "result_matrix_accuracy")

    efficiency_table = forgetting_df.sort_values(["model_key", "train_dataset", "mean_efficiency_ratio"], ascending=[True, True, False])
    save_table(efficiency_table, "efficiency_table")

    recipe_df = derive_practitioner_recipe(forgetting_df)
    save_table(recipe_df, "practitioner_recipe")

    bootstrap_rows = []
    comparisons = []
    for model_key in flat_df["model_key"].dropna().unique():
        fam = family_for_model(model_key)
        if fam == "qwen3_5":
            comparisons.extend([
                (model_key, "softmax_only", "all_layers"),
                (model_key, "gdn_only", "softmax_only"),
            ])
        elif fam == "falcon_h1":
            comparisons.extend([
                (model_key, "attention_only", "all_eligible"),
                (model_key, "ssm_only", "attention_only"),
            ])

    for model_key, cond_a, cond_b in comparisons:
        for train_dataset in flat_df["train_dataset"].dropna().unique():
            for benchmark in flat_df["benchmark"].dropna().unique():
                sub_a = flat_df[(flat_df["model_key"] == model_key) & (flat_df["condition"] == cond_a) & (flat_df["train_dataset"] == train_dataset) & (flat_df["benchmark"] == benchmark)]
                sub_b = flat_df[(flat_df["model_key"] == model_key) & (flat_df["condition"] == cond_b) & (flat_df["train_dataset"] == train_dataset) & (flat_df["benchmark"] == benchmark)]
                if sub_a.empty or sub_b.empty:
                    continue
                detail_a = sub_a["detail_path"].dropna().iloc[0]
                detail_b = sub_b["detail_path"].dropna().iloc[0]
                stats = paired_bootstrap(detail_a, detail_b, n_boot=1000)
                bootstrap_rows.append({
                    "model_key": model_key,
                    "train_dataset": train_dataset,
                    "benchmark": benchmark,
                    "condition_a": cond_a,
                    "condition_b": cond_b,
                    **stats,
                })

    bootstrap_df = pd.DataFrame(bootstrap_rows)
    if not bootstrap_df.empty:
        save_table(bootstrap_df, "paired_bootstrap_comparisons")

    for model_key in flat_df["model_key"].dropna().unique():
        make_grouped_bar(flat_df, model_key, FIGURES_DIR / f"{model_key}__mean_delta_bar")
        make_scatter_pareto(forgetting_df, model_key, FIGURES_DIR / f"{model_key}__pareto_scatter")
        for train_dataset in flat_df["train_dataset"].dropna().unique():
            make_heatmap(flat_df, model_key, train_dataset, FIGURES_DIR / f"{model_key}__{train_dataset}__heatmap")
            make_loss_overlay(all_payloads, model_key, train_dataset, FIGURES_DIR / f"{model_key}__{train_dataset}__loss_overlay")

            fam = family_for_model(model_key)
            if fam == "qwen3_5":
                radar_conditions = [c for c in ["all_layers", "softmax_only", "gdn_only", "mlp_only"] if c in flat_df["condition"].unique()]
            else:
                radar_conditions = [c for c in ["all_eligible", "attention_only", "ssm_only", "mlp_only"] if c in flat_df["condition"].unique()]
            make_radar(flat_df, model_key, train_dataset, radar_conditions, FIGURES_DIR / f"{model_key}__{train_dataset}__radar")

    summary_json = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "experiment_preset": EXPERIMENT_PRESET,
        "models_to_run": MODELS_TO_RUN,
        "datasets_to_use": DATASETS_TO_USE,
        "num_payloads": len(all_payloads),
        "num_flat_rows": len(flat_df),
        "best_mean_accuracy_rows": (
            forgetting_df.sort_values("mean_accuracy", ascending=False).head(20).to_dict(orient="records")
            if not forgetting_df.empty else []
        ),
        "recipe_rows": recipe_df.to_dict(orient="records") if not recipe_df.empty else [],
    }
    save_text(SUMMARY_DIR / "paper3_summary.json", json.dumps(summary_json, indent=2, default=json_default))

    print("✅ Analysis complete.")
    print("Saved to:")
    print(" - tables:", TABLES_DIR)
    print(" - figures:", FIGURES_DIR)
    print(" - summary JSON:", SUMMARY_DIR / "paper3_summary.json")

    display(result_matrix.head(20))
    display(forgetting_df.head(20))
    display(recipe_df)


# Recommended execution order

## For reproducing the paper results

**Phase 1 — Discovery (~30 min)**
1. Run cells 0–4 (install, configure `ROOT_DIR`, config)
2. Run cells 5–9 (discovery + verification)
3. **STOP** — inspect module trees in `MODULE_TREES_DIR` before continuing

**Phase 2 — Data prep (~15 min)**  
4. Run cells 10–12 (datasets)

**Phase 3 — Kill switch (~5h GPU)**
5. Run cells 14–15 (eval + training helpers)
6. In cell 16, execute: `kill_results = run_kill_switch_block()`
7. **Read the verdict** — 🔴 stop, 🟡 proceed cautiously, 🟢 full speed

**Phase 4 — Full grid (~25h GPU)**
8. In cell 16: `grid_results = run_experiment_grid()`
9. *(Checkpoint-safe: if runtime disconnects, re-run cells 0–4 then cell 16. Completed runs auto-skip.)*

**Phase 5 — Analysis**
10. Run cells 17–19 (Layer Card, dynamics)
11. Run cell 21 (figures, LaTeX tables, summary JSON)

**Phase 6 — Optional extras**
- Multi-seed: `ENABLE_MULTI_SEED_CORE = True` → re-run cell 16
- Rank ablation: `ENABLE_RANK_ABLATION = True` → re-run cell 16

## Hardware requirements
- **GPU with ≥24GB VRAM** (e.g., L4, RTX 4090, A5000, A100)
- **Avoid T4 16GB** — too tight for Falcon-H1 with eval batches
- Tested on NVIDIA L4 (Colab Pro) and RTX 4090 (RunPod)
